In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/Managers.csv'

In [ ]:
# =====================================================
# LOAD Managers
# =====================================================
df = pd.read_csv(path, encoding="utf-8-sig", dtype=str)
df
out = df.copy()
out

,ManagerID,ManagerName,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment
0,MGR0001,Ivan M.,Sales A,Operations,2023-12-27,NaN,Active,Kyiv,Ukraine,Full-time,44500.0,23.0,294000.0,Strong
1,MGR0002,Natalia H.,Sales B,Sales,2022-03-05,2024-05-25,Terminated,Lviv,Ukraine,Part-time,74200.0,6.0,268000.0,Top
2,MGR0003,Oksana B.,Retention,Customer Success,2022-07-14,NaN,Active,Warsaw,Poland,Part-time,64700.0,17.7,529000.0,Average
3,MGR0004,Vasyl F.,Sales A,Sales,2021-05-09,NaN,Active,Berlin,Germany,Contract,64300.0,23.8,381000.0,Strong
4,MGR0005,Иларион Д.,Sales B,Sales,2022-05-18,2025-09-13,Terminated,Prague,Czech Republic,Full-time,45100.0,17.6,457000.0,Weak
5,MGR0006,John З.,Retention,Customer Success,2023-11-19,NaN,Active,Kyiv,Ukraine,Part-time,49100.0,13.5,380000.0,Weak
6,MGR0007,Орест К.,Sales A,Sales,2022-11-09,NaN,Active,Lviv,Ukraine,Contract,48000.0,22.1,506000.0,Average
7,MGR0008,Василь Г.,Sales B,Sales,2021-11-15,2024-02-03,Terminated,Warsaw,Poland,Contract,56600.0,22.3,310000.0,Top


Data *Audit*

In [ ]:
out.shape

(8, 14)

ManagerID - Primary Key and Foreign key.

In [ ]:
is_unique = out['ManagerID'].is_unique
print(is_unique)

True


Checking for missing values

In [ ]:
out.isnull().sum()

,0
ManagerID,0
ManagerName,0
Team,0
Department,0
HireDate,0
TerminationDate,5
EmploymentStatus,0
Region,0
Country,0
EmploymentType,0


пропущені значення є тільки в стовбці TerminationDate. Це говорить про те що більшість менеджерів не звільнились, продовжують працювати. Залишаємо nan

In [ ]:
out.dtypes

,0
ManagerID,object
ManagerName,object
Team,object
Department,object
HireDate,object
TerminationDate,object
EmploymentStatus,object
Region,object
Country,object
EmploymentType,object


всі строки таблиці Managers мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як float/int, string, date, category, які добре працюють та займають менше оперативної пам'яті.

##Checking for duplicates

In [ ]:
total_duplicates = out.duplicated().sum()
total_duplicates

np.int64(0)

дублікатів в таблиці Managers немає.

Part 2 Data Cleaning

Data Types and Dates

In [ ]:
def clean_money(x):
    if pd.isna(x): return np.nan
    x = re.sub(r"[₴грн\s]", "", str(x)).replace(",", ".")
    x = re.sub(r"[^0-9.]", "", x)
    try: return float(x)
    except: return np.nan

In [ ]:
out['ManagerID'] = out['ManagerID'].astype(str)
out['ManagerName'] = out['ManagerName'].astype(str)
out['Team'] = out['Team'].astype(str)
out['Department'] = out['Department'].astype(str)
out['EmploymentStatus'] = out['EmploymentStatus'].astype(str)
out['Country'] = out['Country'].astype(str)
out['Region'] = out['Region'].astype(str)
out['EmploymentType'] = out['EmploymentType'].astype(str)
out['MonthlySalary'] = out['MonthlySalary'].apply(clean_money)
out['BonusPercent'] = out['BonusPercent'].astype(float)
out['MonthlySalesTarget'] = out['MonthlySalesTarget'].apply(clean_money)
out['ManagerPerformanceSegment'] = out['ManagerPerformanceSegment'].astype(str)


In [21]:
!pip install transliterate
from transliterate import translit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.3 MB/s eta 0:00:00


In [ ]:
out['Manager_en'] = out['ManagerName'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)

In [ ]:
out['Manager_en'].unique()

array(['Ivan M.', 'Natalia H.', 'Oksana B.', 'Vasyl F.', 'Ilarion D.',
       'John Z.', 'Orest K.', "Vasil' G."], dtype=object)

In [ ]:
out

,ManagerID,ManagerName,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment,Manager_en
0,MGR0001,Ivan M.,Sales A,Operations,2023-12-27,NaN,Active,Kyiv,Ukraine,Full-time,44500.0,23.0,294000.0,Strong,Ivan M.
1,MGR0002,Natalia H.,Sales B,Sales,2022-03-05,2024-05-25,Terminated,Lviv,Ukraine,Part-time,74200.0,6.0,268000.0,Top,Natalia H.
2,MGR0003,Oksana B.,Retention,Customer Success,2022-07-14,NaN,Active,Warsaw,Poland,Part-time,64700.0,17.7,529000.0,Average,Oksana B.
3,MGR0004,Vasyl F.,Sales A,Sales,2021-05-09,NaN,Active,Berlin,Germany,Contract,64300.0,23.8,381000.0,Strong,Vasyl F.
4,MGR0005,Иларион Д.,Sales B,Sales,2022-05-18,2025-09-13,Terminated,Prague,Czech Republic,Full-time,45100.0,17.6,457000.0,Weak,Ilarion D.
5,MGR0006,John З.,Retention,Customer Success,2023-11-19,NaN,Active,Kyiv,Ukraine,Part-time,49100.0,13.5,380000.0,Weak,John Z.
6,MGR0007,Орест К.,Sales A,Sales,2022-11-09,NaN,Active,Lviv,Ukraine,Contract,48000.0,22.1,506000.0,Average,Orest K.
7,MGR0008,Василь Г.,Sales B,Sales,2021-11-15,2024-02-03,Terminated,Warsaw,Poland,Contract,56600.0,22.3,310000.0,Top,Vasil' G.


In [ ]:
out.drop(columns=['ManagerName'], inplace=True)

In [ ]:
out['HireDate'] = pd.to_datetime(out['HireDate'], dayfirst=True, errors='coerce')
out['TerminationDate'] = pd.to_datetime(out['TerminationDate'], dayfirst=True, errors='coerce')

/tmp/ipykernel_3718/1489658609.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  out['HireDate'] = pd.to_datetime(out['HireDate'], dayfirst=True, errors='coerce')
/tmp/ipykernel_3718/1489658609.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  out['TerminationDate'] = pd.to_datetime(out['TerminationDate'], dayfirst=True, errors='coerce')


In [ ]:
invalid_date_logic = out[out['TerminationDate'] < out['HireDate']]
invalid_date_logic

,ManagerID,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment,Manager_en


In [ ]:
invalid_monthly_salary = out[out['MonthlySalary'] < 0]
invalid_monthly_salary

,ManagerID,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment,Manager_en


In [ ]:
invalid_monthly_target = out[out['MonthlySalesTarget'] < 0]
invalid_monthly_target

,ManagerID,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment,Manager_en


In [ ]:
invalid_bonus_percent = out[(out['BonusPercent'] < 0) | (out['BonusPercent'] > 100)]
invalid_bonus_percent

,ManagerID,Team,Department,HireDate,TerminationDate,EmploymentStatus,Region,Country,EmploymentType,MonthlySalary,BonusPercent,MonthlySalesTarget,ManagerPerformanceSegment,Manager_en


In [ ]:
# =====================================================
# EXPORT
# =====================================================
out.to_csv("Managers_cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out.shape)

DONE: (8, 14)


In [ ]:
path1 = '/content/drive/MyDrive/Courses.csv'

In [ ]:
# =====================================================
# LOAD Courses
# =====================================================
df1 = pd.read_csv(path1, encoding="utf-8-sig", dtype=str)
df1
out1 = df1.copy()
out1

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BasePrice,BaseCurrency,Language,LaunchDate,IsActive,TeacherName,PlannedSeats,VariableCostPerStudent,CoursePopularitySegment
0,CRS0001,AI Automation,AI Automation,Hybrid,Intermediate,12,37600.0,UAH,UA,2021-09-29,True,Tadeusz Мельникова,50,6768.0,Popular
1,CRS0002,Data Analytics,Data Analytics,Offline,Advanced,9,36800.0,UAH,UA,2023-02-11,True,Андрій Михалюк,59,6624.0,Standard
2,CRS0003,DevOps,DevOps,Offline,Intermediate,21,52400.0,UAH,UA,2024-07-05,True,Julian Негода,117,9432.0,Popular
3,CRS0004,Excel,Excel,Offline,Beginner,13,8700.0,UAH,UA,2024-03-28,True,Аліна Andryszczyk,77,1566.0,Standard
4,CRS0005,Front-End,Front-End,Online,Beginner,6,28600.0,UAH,UA,2023-05-11,True,Фирс Кононов,26,5148.0,Flagship
5,CRS0006,FullStack Java,Full Stack,Online,Intermediate,10,38100.0,UAH,UA,2023-03-30,True,Donna Tabak,25,6858.0,Standard
6,CRS0007,FullStack Python,Full Stack,Online,Advanced,27,42100.0,UAH,UA,2022-12-27,True,Bailey Волков,61,7578.0,Niche
7,CRS0008,IT Start (Free),Introductory,Online,Beginner,3,0.0,UAH,UA,2023-01-05,True,Andrew Ершов,108,194.31,Flagship
8,CRS0009,Java,Java,Hybrid,Intermediate,30,36700.0,UAH,UA,2024-03-27,True,Давид Абрагамовська,39,6606.0,Niche
9,CRS0010,Python,Python,Offline,Intermediate,29,28100.0,UAH,UA,2022-09-29,True,Варфоломей Арсенич,103,5058.0,Standard


Data *Audit* Courses

In [ ]:
out1.shape

(17, 15)

CourseID - Primary Key and Foreign key.

In [ ]:
is_unique_1 = out1['CourseID'].is_unique
print(is_unique_1)

True


Checking for missing values

In [ ]:
out1.isnull().sum()

,0
CourseID,0
CourseName,0
CourseCategory,0
DeliveryFormat,0
DifficultyLevel,0
DurationWeeks,0
BasePrice,0
BaseCurrency,0
Language,0
LaunchDate,0


In [ ]:
out1.dtypes

,0
CourseID,object
CourseName,object
CourseCategory,object
DeliveryFormat,object
DifficultyLevel,object
DurationWeeks,object
BasePrice,object
BaseCurrency,object
Language,object
LaunchDate,object


всі строки таблиці Courses мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як float/int, string, date, category, які добре працюють та займають менше оперативної пам'яті.

Checking for duplicates

In [ ]:
total_duplicates_1 = out1.duplicated().sum()
total_duplicates_1

np.int64(0)

дублікатів в таблиці courses немає

Part 2 Data Cleaning

Data Types and Dates

In [ ]:
out1['CourseID'] = out1['CourseID'].astype(str)
out1['CourseName'] = out1['CourseName'].astype(str)
out1['CourseCategory'] = out1['CourseCategory'].astype(str)
out1['DeliveryFormat'] = out1['DeliveryFormat'].astype(str)
out1['DifficultyLevel'] = out1['DifficultyLevel'].astype(str)
out1['DurationWeeks'] = out1['DurationWeeks'].astype(int)
out1['BaseCurrency'] = out1['BaseCurrency'].astype(str)
out1['Language'] = out1['Language'].astype(str)
out1['IsActive'] = out1['IsActive'].astype(bool)
out1['TeacherName'] =out1['TeacherName'].astype(str)
out1['PlannedSeats'] = out1['PlannedSeats'].astype(int)
out1['CoursePopularitySegment'] = out1['CoursePopularitySegment'].astype(str)
out1['LaunchDate'] = pd.to_datetime(out1['LaunchDate'], dayfirst=True, errors='coerce')

/tmp/ipykernel_3718/3750066964.py:13: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  out1['LaunchDate'] = pd.to_datetime(out1['LaunchDate'], dayfirst=True, errors='coerce')


In [ ]:
out1["Clean_Base_Price"] = out1['BasePrice'].apply(clean_money).fillna(0)
out1["Clean_VariableCostPerStudent"] = out1['VariableCostPerStudent'].apply(clean_money).fillna(0)

In [ ]:
out1

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BasePrice,BaseCurrency,Language,LaunchDate,IsActive,PlannedSeats,VariableCostPerStudent,CoursePopularitySegment,TeacherName_en,Clean_Base_Price,Clean_VariableCostPerStudent
0,CRS0001,AI Automation,AI Automation,Hybrid,Intermediate,12,37600.0,UAH,UA,2021-09-29,True,50,6768.00,Popular,Tadeusz Mel'nikova,37600.0,6768.00
1,CRS0002,Data Analytics,Data Analytics,Offline,Advanced,9,36800.0,UAH,UA,2023-02-11,True,59,6624.00,Standard,Andrіj Mihaljuk,36800.0,6624.00
2,CRS0003,DevOps,DevOps,Offline,Intermediate,21,52400.0,UAH,UA,2024-07-05,True,117,9432.00,Popular,Julian Negoda,52400.0,9432.00
3,CRS0004,Excel,Excel,Offline,Beginner,13,8700.0,UAH,UA,2024-03-28,True,77,1566.00,Standard,Alіna Andryszczyk,8700.0,1566.00
4,CRS0005,Front-End,Front-End,Online,Beginner,6,28600.0,UAH,UA,2023-05-11,True,26,5148.00,Flagship,Firs Kononov,28600.0,5148.00
5,CRS0006,FullStack Java,Full Stack,Online,Intermediate,10,38100.0,UAH,UA,2023-03-30,True,25,6858.00,Standard,Donna Tabak,38100.0,6858.00
6,CRS0007,FullStack Python,Full Stack,Online,Advanced,27,42100.0,UAH,UA,2022-12-27,True,61,7578.00,Niche,Bailey Volkov,42100.0,7578.00
7,CRS0008,IT Start (Free),Introductory,Online,Beginner,3,0.0,UAH,UA,2023-01-05,True,108,194.31,Flagship,Andrew Ershov,0.0,194.31
8,CRS0009,Java,Java,Hybrid,Intermediate,30,36700.0,UAH,UA,2024-03-27,True,39,6606.00,Niche,David Abragamovs'ka,36700.0,6606.00
9,CRS0010,Python,Python,Offline,Intermediate,29,28100.0,UAH,UA,2022-09-29,True,103,5058.00,Standard,Varfolomej Arsenich,28100.0,5058.00


In [ ]:
out1['TeacherName_en'] = out1['TeacherName'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)

In [ ]:
out1['TeacherName_en'].unique()

array(["Tadeusz Mel'nikova", 'Andrіj Mihaljuk', 'Julian Negoda',
       'Alіna Andryszczyk', 'Firs Kononov', 'Donna Tabak',
       'Bailey Volkov', 'Andrew Ershov', "David Abragamovs'ka",
       'Varfolomej Arsenich', 'Aurelia Vaschenko', 'Mariusz Turova',
       'Klavdіja Smik', 'Curtis Їzhak', 'Michał Haas', 'Mark Vasilenko',
       'Galaktion Averchenko'], dtype=object)

In [ ]:
out1.drop(columns=['TeacherName', 'BasePrice', 'VariableCostPerStudent'], inplace=True)

In [ ]:
invalid_base_price = out1[out1['Clean_Base_Price'] < 0]
invalid_base_price

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BaseCurrency,Language,LaunchDate,IsActive,PlannedSeats,CoursePopularitySegment,Clean_Base_Price,Clean_VariableCostPerStudent,TeacherName_en


In [ ]:
invalid_weeks = out1[out1['DurationWeeks'] < 0]
invalid_weeks

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BaseCurrency,Language,LaunchDate,IsActive,PlannedSeats,CoursePopularitySegment,Clean_Base_Price,Clean_VariableCostPerStudent,TeacherName_en


In [ ]:
invalid_seats = out1[out1['PlannedSeats'] <= 0]
invalid_seats

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BaseCurrency,Language,LaunchDate,IsActive,PlannedSeats,CoursePopularitySegment,Clean_Base_Price,Clean_VariableCostPerStudent,TeacherName_en


In [ ]:
invalid_cost = out1[out1['Clean_VariableCostPerStudent'] < 0]
invalid_cost

,CourseID,CourseName,CourseCategory,DeliveryFormat,DifficultyLevel,DurationWeeks,BaseCurrency,Language,LaunchDate,IsActive,PlannedSeats,CoursePopularitySegment,Clean_Base_Price,Clean_VariableCostPerStudent,TeacherName_en


In [ ]:
out1['BaseCurrency'].unique()

array(['UAH'], dtype=object)

In [ ]:
# =====================================================
# EXPORT
# =====================================================
out1.to_csv("Courses_Cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out1.shape)

DONE: (17, 15)


In [ ]:
path2 = '/content/drive/MyDrive/Enrollments.csv'

In [ ]:
# =====================================================
# LOAD Enrollments
# =====================================================
df2 = pd.read_csv(path2, encoding="utf-8-sig", dtype=str)
df2
out2 = df2.copy()
out2

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,EnrollmentStatus,AgreedPrice,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate
0,ENR000001,STU000001,LEAD0000018,CRS0008,MGR0002,2024-01-03,COH00090,2024-01-14,2024-02-04,NaN,Dropped,0.0,0,USD,Free,google,NaN,NaN,False,NaN
1,ENR000002,STU000002,LEAD0000061,CRS0001,MGR0008,2024-01-09,COH00001,2024-01-17,2024-04-10,NaN,Dropped,3690.8,0,PLN,Full,google,NaN,NaN,False,NaN
2,ENR000003,STU000003,LEAD0000087,CRS0012,MGR0004,2024-01-03,COH00152,2024-02-14,2024-07-24,2024-08-01,Completed,662.9,15,USD,Full,meta,NaN,NaN,True,NaN
3,ENR000004,STU000004,LEAD0000123,CRS0015,MGR0005,2024-01-15,COH00198,2024-02-12,2024-03-04,2024-03-01,Completed,0.0,0,EUR,Free,telegram,NaN,NaN,False,NaN
4,ENR000005,STU000005,LEAD0000301,CRS0017,MGR0007,2024-01-04,COH00231,2024-01-20,2024-07-06,2024-07-02,Completed,22320.0,20,UAH,Installments,google,NaN,NaN,True,2024-07-19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12156,ENR012157,STU011673,LEAD0046280,CRS0006,MGR0006,2026-01-12,COH00078,2025-11-08,2026-01-17,NaN,Active,38100.0,0,UAH,Full,chatgpt.com,NaN,NaN,False,NaN
12157,ENR012158,STU011674,LEAD0048494,CRS0008,MGR0003,2026-01-11,COH00115,2025-12-14,2026-01-04,NaN,Active,0.0,0,PLN,Free,google,NaN,NaN,False,NaN
12158,ENR012159,STU010968,LEAD0048673,CRS0017,MGR0004,2026-01-04,COH00242,2025-12-13,2026-05-30,NaN,Cancelled,27900.0,0,UAH,Installments,referral,2026-01-08,Dissatisfied,False,NaN
12159,ENR012160,STU011675,LEAD0046995,CRS0005,MGR0003,2026-01-11,COH00065,2025-11-18,2025-12-30,2026-01-01,Completed,20020.0,30,UAH,Installments,dou,NaN,NaN,True,2026-01-05


Data audit

In [ ]:
out2.shape

(12161, 20)

EnrollmentID - Primary Key; StudentID, LeadID, CourseID, Manager - Foreign Key

In [ ]:
is_unique_2 = out2['EnrollmentID'].is_unique
print(is_unique_2)

True


Checking for missing values

In [ ]:
out2.isnull().sum()

,0
EnrollmentID,0
StudentID,0
LeadID,0
CourseID,0
ManagerID,0
EnrollmentDate,0
CohortID,0
CourseStartDate,0
ExpectedEndDate,0
ActualCompletionDate,6520


пропущені значення є в стовбцях ActualCompletionDate, CancellationDate, CancellationReason and CertificateIssuedDate. При обробці пропущених значень використовуватиметься метод помічання (flag) та метод заповнення (imput). Так можна бистро порахувати відсоток відмін та фідфільтрувати студентів які завершили курс та отримали сертифікат в Power BI.

In [ ]:
out2.dtypes

,0
EnrollmentID,object
StudentID,object
LeadID,object
CourseID,object
ManagerID,object
EnrollmentDate,object
CohortID,object
CourseStartDate,object
ExpectedEndDate,object
ActualCompletionDate,object


всі строки таблиці  Enrollments мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як float/int, string, date, category, які добре працюють та займають менше оперативної пам'яті.

Checking for duplicates

In [ ]:
total_duplicates_2 = out2.duplicated().sum()
total_duplicates_2

np.int64(0)

Дублікатів в таблиці Enrollments немає.

Part 2 Data Cleaning

Data Types and Dates

In [ ]:
out2['AcquisitionSource'].unique()

array(['google', 'meta', 'telegram', 'email', 'organic', 'referral',
       'dou', 'instagram', 'direct', 'linkedin', 'chatgpt.com',
       'perplexity'], dtype=object)

In [ ]:
out2['EnrollmentID'] = out2['EnrollmentID'].astype(str)
out2['StudentID'] = out2['StudentID'].astype(str)
out2['LeadID'] = out2['LeadID'].astype(str)
out2['CourseID'] = out2['CourseID'].astype(str)
out2['ManagerID'] = out2['ManagerID'].astype(str)
out2['CohortID'] = out2['CohortID'].astype(str)
out2['EnrollmentStatus'] = out2['EnrollmentStatus'].astype(str)
out2['Currency'] = out2['Currency'].astype(str)
out2['PaymentPlan'] = out2['PaymentPlan'].astype(str)
out2['AcquisitionSource'] =out2['AcquisitionSource'].astype(str)
out2['CancellationReason'] = out2['CancellationReason'].astype(str)
out2['CertificateEligible'] = out2['CertificateEligible'].astype(bool)




In [ ]:
from dateutil import parser

def parse_hybrid_date(val):
    if pd.isna(val): return pd.NaT
    try:
        return parser.parse(str(val), dayfirst=True)
    except:
        return pd.NaT

out2['CourseStartDate'] = out2['CourseStartDate'].apply(parse_hybrid_date)
out2['ActualCompletionDate'] = out2['ActualCompletionDate'].apply(parse_hybrid_date)
out2['ExpectedEndDate'] = out2['ExpectedEndDate'].apply(parse_hybrid_date)
out2['EnrollmentDate'] = out2['EnrollmentDate'].apply(parse_hybrid_date)
out2['CancellationDate'] = out2['CancellationDate'].apply(parse_hybrid_date)
out2['CertificateIssuedDate'] = out2['CertificateIssuedDate'].apply(parse_hybrid_date)


In [ ]:
out2["AgreedPrice"] = out2["AgreedPrice"].apply(clean_money)
out2["DiscountPercent"] = out2['DiscountPercent'].apply(clean_money)

Handling missing values

In [ ]:
out2['is_canceled'] = out2['CancellationDate'].notna().astype(int)

In [ ]:
out2['CancellationReason'] = out2['CancellationReason'].fillna('Not Cancelled')

In [ ]:
out2['is_certified'] = out2['CertificateIssuedDate'].notna().astype(int)

In [ ]:
invalid_start_dates = out2[out2['CourseStartDate'] < out2['EnrollmentDate']]
invalid_start_dates

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified
10912,ENR010913,STU010518,LEAD0043211,CRS0006,MGR0006,2025-11-13,COH00078,2025-11-08,2026-01-17,NaT,...,5.0,UAH,Full,telegram,2025-12-10,Health,True,NaT,1,0
10933,ENR010934,STU010539,LEAD0043108,CRS0010,MGR0006,2025-11-14,COH00138,2025-11-12,2026-06-03,NaT,...,0.0,UAH,Full,instagram,NaT,nan,True,NaT,0,0
10968,ENR010969,STU010571,LEAD0044581,CRS0010,MGR0006,2025-11-14,COH00138,2025-11-12,2026-06-03,NaT,...,15.0,UAH,Full,google,NaT,nan,True,NaT,0,0
10992,ENR010993,STU010593,LEAD0042670,CRS0005,MGR0004,2025-11-20,COH00065,2025-11-18,2025-12-30,NaT,...,15.0,UAH,Installments,meta,NaT,nan,True,NaT,0,0
10999,ENR011000,STU010600,LEAD0042770,CRS0006,MGR0007,2025-11-20,COH00078,2025-11-08,2026-01-17,NaT,...,0.0,UAH,Full,chatgpt.com,2026-01-11,Job change,True,NaT,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12156,ENR012157,STU011673,LEAD0046280,CRS0006,MGR0006,2026-01-12,COH00078,2025-11-08,2026-01-17,NaT,...,0.0,UAH,Full,chatgpt.com,NaT,nan,True,NaT,0,0
12157,ENR012158,STU011674,LEAD0048494,CRS0008,MGR0003,2026-01-11,COH00115,2025-12-14,2026-01-04,NaT,...,0.0,PLN,Free,google,NaT,nan,True,NaT,0,0
12158,ENR012159,STU010968,LEAD0048673,CRS0017,MGR0004,2026-01-04,COH00242,2025-12-13,2026-05-30,NaT,...,0.0,UAH,Installments,referral,2026-01-08,Dissatisfied,True,NaT,1,0
12159,ENR012160,STU011675,LEAD0046995,CRS0005,MGR0003,2026-01-11,COH00065,2025-11-18,2025-12-30,2026-01-01,...,30.0,UAH,Installments,dou,NaT,nan,True,2026-01-05,0,1


In [ ]:
invalid_end_dates = out2[out2['ExpectedEndDate'] < out2['CourseStartDate']]
invalid_end_dates

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified


In [ ]:
invalid_base_price_1 = out2[out2['AgreedPrice'] < 0]
invalid_base_price_1

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified


In [ ]:
invalid_completion_dates = out2[out2['CourseStartDate'] > out2['ActualCompletionDate']]
invalid_completion_dates

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified


In [ ]:
invalid_certificate_date = out2[out2['CertificateIssuedDate'] < out2['ActualCompletionDate']]
invalid_certificate_date

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified


In [ ]:
invalid_percent = out2[(out2['DiscountPercent'] < 0) | (out2['DiscountPercent'] > 100)]
invalid_percent

,EnrollmentID,StudentID,LeadID,CourseID,ManagerID,EnrollmentDate,CohortID,CourseStartDate,ExpectedEndDate,ActualCompletionDate,...,DiscountPercent,Currency,PaymentPlan,AcquisitionSource,CancellationDate,CancellationReason,CertificateEligible,CertificateIssuedDate,is_canceled,is_certified


In [ ]:
# =====================================================
# EXPORT
# =====================================================
out2.to_csv("Enrollments_Cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out2.shape)

DONE: (12161, 22)


In [ ]:
path3 = '/content/drive/MyDrive/Cohorts.csv'

In [ ]:
# =====================================================
# LOAD Cohorts
# =====================================================
df3 = pd.read_csv(path3, encoding="utf-8-sig", dtype=str)
df3
out3 = df3.copy()
out3

,CohortID,CourseID,CohortName,StartDate,PlannedEndDate,TeacherName,DeliveryFormat,PlannedSeats,ActualEnrollments,CohortStatus
0,COH00001,CRS0001,AI Automation #01 2024-01,2024-01-17,2024-04-10,Tadeusz Мельникова,Hybrid,50,4,Completed
1,COH00002,CRS0001,AI Automation #02 2024-03,2024-03-13,2024-06-05,Tadeusz Мельникова,Hybrid,50,49,Completed
2,COH00003,CRS0001,AI Automation #03 2024-05,2024-05-08,2024-07-31,Tadeusz Мельникова,Hybrid,50,60,Completed
3,COH00004,CRS0001,AI Automation #04 2024-07,2024-07-03,2024-09-25,Tadeusz Мельникова,Hybrid,50,54,Completed
4,COH00005,CRS0001,AI Automation #05 2024-08,2024-08-28,2024-11-20,Tadeusz Мельникова,Hybrid,50,37,Completed
...,...,...,...,...,...,...,...,...,...,...
237,COH00238,CRS0017,QA Engineer #08 2025-04,2025-04-05,2025-09-20,Галактион Аверченко,Hybrid,114,47,Completed
238,COH00239,CRS0017,QA Engineer #09 2025-06,2025-06-07,2025-11-22,Галактион Аверченко,Hybrid,114,33,Completed
239,COH00240,CRS0017,QA Engineer #10 2025-08,2025-08-09,2026-01-24,Галактион Аверченко,Hybrid,114,22,In Progress
240,COH00241,CRS0017,QA Engineer #11 2025-10,2025-10-11,2026-03-28,Галактион Аверченко,Hybrid,114,30,In Progress


CohortID - Primary Key; CourseID - Foreign Key

In [ ]:
is_unique_3 = out3['CohortID'].is_unique
print(is_unique_3)

True


Checking for missing values

In [ ]:
out3.isnull().sum()

,0
CohortID,0
CourseID,0
CohortName,0
StartDate,0
PlannedEndDate,0
TeacherName,0
DeliveryFormat,0
PlannedSeats,0
ActualEnrollments,0
CohortStatus,0


пропущених значень в таблиці Cohorts немає.

In [ ]:
out3.dtypes

,0
CohortID,object
CourseID,object
CohortName,object
StartDate,object
PlannedEndDate,object
TeacherName,object
DeliveryFormat,object
PlannedSeats,object
ActualEnrollments,object
CohortStatus,object


всі строки таблиці Cohorts мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як float/int, string, date, category, які добре працюють та займають менше оперативної пам'яті.

Checking for duplicates

In [ ]:
total_duplicates_3 = out3.duplicated().sum()
total_duplicates_3

np.int64(0)

дублікатів в таблиці cohorts немає.

Part 2 Data Cleaning

Data Types and Dates

In [ ]:
out3['CohortID'] = out3['CohortID'].astype(str)
out3['CourseID'] = out3['CourseID'].astype(str)
out3['CohortName'] = out3['CohortName'].astype(str)
out3['TeacherName'] = out3['TeacherName'].astype(str)
out3['DeliveryFormat'] = out3['DeliveryFormat'].astype(str)
out3['PlannedSeats'] = out3['PlannedSeats'].astype(int)
out3['ActualEnrollments'] = out3['ActualEnrollments'].astype(int)
out3['CohortStatus'] = out3['CohortStatus'].astype(str)

In [ ]:
from dateutil import parser

def parse_hybrid_date(val):
    if pd.isna(val): return pd.NaT
    try:
        return parser.parse(str(val), dayfirst=True)
    except:
        return pd.NaT

out3['StartDate'] = out3['StartDate'].apply(parse_hybrid_date)
out3['PlannedEndDate'] = out3['PlannedEndDate'].apply(parse_hybrid_date)

In [ ]:
invalid_cohorts_dates = out3[out3['StartDate'] > out3['PlannedEndDate']]
invalid_cohorts_dates

,CohortID,CourseID,CohortName,StartDate,PlannedEndDate,TeacherName,DeliveryFormat,PlannedSeats,ActualEnrollments,CohortStatus


In [ ]:
invalid_planned_seats = out3[out3['PlannedSeats'] <= 0]
invalid_planned_seats

,CohortID,CourseID,CohortName,StartDate,PlannedEndDate,TeacherName,DeliveryFormat,PlannedSeats,ActualEnrollments,CohortStatus


In [ ]:
invalid_actual_enrollments = out3[out3['ActualEnrollments'] < 0]
invalid_actual_enrollments




,CohortID,CourseID,CohortName,StartDate,PlannedEndDate,TeacherName,DeliveryFormat,PlannedSeats,ActualEnrollments,CohortStatus


In [ ]:
out3['TeacherName_en'] = out3['TeacherName'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)

In [ ]:
out3['TeacherName_en'].unique()

array(["Tadeusz Mel'nikova", 'Andrіj Mihaljuk', 'Julian Negoda',
       'Alіna Andryszczyk', 'Firs Kononov', 'Donna Tabak',
       'Bailey Volkov', 'Andrew Ershov', "David Abragamovs'ka",
       'Varfolomej Arsenich', 'Aurelia Vaschenko', 'Mariusz Turova',
       'Klavdіja Smik', 'Curtis Їzhak', 'Michał Haas', 'Mark Vasilenko',
       'Galaktion Averchenko'], dtype=object)

In [ ]:
out3.drop(columns=['TeacherName'], inplace=True)

In [ ]:
# =====================================================
# EXPORT
# =====================================================
out3.to_csv("Cohorts_Cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out3.shape)

DONE: (242, 10)


In [19]:
path4 = '/content/drive/MyDrive/Leads.csv'

In [20]:
# =====================================================
# LOAD Leads
# =====================================================
df4 = pd.read_csv(path4, encoding="utf-8-sig", dtype=str)
df4
out4 = df4.copy()
out4

,LeadID,CreatedAt,UpdatedAt,FirstContactAt,ConvertedAt,FullName,Email,Phone,Country,Region,...,LandingPage,DeviceType,ReferrerDomain,FirstTouchSource,LastTouchSource,ExpectedCoursePrice,ExpectedCurrency,DiscountRequested,LostReason,IsDuplicateCandidate
0,LEAD0000001,2024-01-01 00:00:01,2024-01-03 07:27:16,2024-01-01 00:27:16,NaN,NaN,artur.поляков1127@icloud.com,+380815156336,Ukraine,Kyiv Oblast,...,/ua/start,Mobile,prog.academy,referral,referral,0.0,UAH,False,NaN,False
1,LEAD0000002,2024-01-01 00:00:46,2024-01-04 10:40:33,2024-01-01 19:40:33,NaN,Katherine Сорокин,katherine.сорокин5692@gmail.com,+380113586793,Ukraine,Lviv Oblast,...,/ua/ai-automation,Mobile,NaN,direct,direct,21750.0,UAH,True,NaN,False
2,LEAD0000003,2024-01-01 00:09:30,01/01/2024,2024-01-01 08:56:19,NaN,Marie Małycha,marie.małycha4491@gmail.com,+380983031290,Ukraine,Kharkiv Oblast,...,/ua/qa,Desktop,t.me,telegram,telegram,28200.0,UAH,True,неправильный номер,False
3,LEAD0000004,2024-01-01 00:11:33,2024-01-02 22:56:50,2024-01-01 03:56:50,NaN,Tobiasz Брагина,tobiasz.брагина7657@gmail.com,+380674161203,Ukraine,Lviv Oblast,...,/ua/frontend,Desktop,t.me,telegram,telegram,28130.0,UAH,False,передумал,False
4,LEAD0000005,2024-01-01 00:21:07,2024-01-02 11:05:10,2024-01-01 11:05:10,NaN,Ruth Архипенко,ruth.архипенко894@outlook.com,+380659994537,Ukraine,Lviv Oblast,...,/ua/ai-automation,Desktop,prog.academy,referral,referral,38850.0,UAH,False,дорого,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51774,LEAD0049538,2025-12-26 22:57:39,2025-12-27 06:54:06,2025-12-27 01:54:06,NaN,Потап Авдєєнко,потап.авдєєнко7591@outlook.com,+380944429375,Ukraine,Lviv Oblast,...,/ua/fullstack,Mobile,t.me,telegram,telegram,39840.0,UAH,True,нет нужного курса,False
51775,LEAD0049593,2025-12-27 16:49:42,2025-12-30 03:43:35,2025-12-28 10:43:35,NaN,Benjamin Mccoy,benjamin.mccoy510@icloud.com,+380428708493,Ukraine,Kyiv Oblast,...,/ua/fullstack,Desktop,chatgpt.com,chatgpt.com,chatgpt.com,41930.0,UAH,False,NaN,False
51776,LEAD0050013D,2024-01-06 01:43:17,2024-01-19 02:52:45,NaN,2024-01-17 13:52:45,MICHAEL АКСЕНОВ,MICHAEL.АКСЕНОВ3791@YAHOO.COM,+44 (19) 1123026,United Kingdom,Manchester Region,...,/ua/ai-automation,Mobile,facebook.com,meta,meta,38880.0,EUR,True,NaN,True
51777,LEAD0020580,2024-11-07 13:09:49,2024-11-10 10:38:39,2024-11-07 17:38:39,NaN,Галина Hodge,галина.hodge4510@outlook.com,+380116235226,Ukraine,--,...,/ua/qa,Mobile,t.me,telegram,Telegram,26490.0,UAH,False,перестал отвечать в чате,False


In [ ]:
out4.columns.to_list()

['LeadID',
 'CreatedAt',
 'UpdatedAt',
 'FirstContactAt',
 'ConvertedAt',
 'FullName',
 'Email',
 'Phone',
 'Country',
 'Region',
 'City',
 'PreferredLanguage',
 'CourseID',
 'CourseNameRaw',
 'ManagerID',
 'ManagerNameRaw',
 'LeadStatus',
 'LeadStage',
 'LeadTemperature',
 'LeadScore',
 'Source',
 'Medium',
 'Campaign',
 'Content',
 'Term',
 'LandingPage',
 'DeviceType',
 'ReferrerDomain',
 'FirstTouchSource',
 'LastTouchSource',
 'ExpectedCoursePrice',
 'ExpectedCurrency',
 'DiscountRequested',
 'LostReason',
 'IsDuplicateCandidate']

LeadID-primary key; ManagerID, CourseID-foreign key.

In [22]:
out4['LeadID'] = out4['LeadID'].astype(str).str.strip()

In [23]:
is_unique_4 = out4['LeadID'].is_unique
is_unique_4


False

Checking for duplicates

In [24]:
total_duplicates_4 = out4.duplicated().sum()
total_duplicates_4

np.int64(279)

In [25]:
out4_cleaned = out4.drop_duplicates(subset=['LeadID'], keep='first')
out4_cleaned

,LeadID,CreatedAt,UpdatedAt,FirstContactAt,ConvertedAt,FullName,Email,Phone,Country,Region,...,LandingPage,DeviceType,ReferrerDomain,FirstTouchSource,LastTouchSource,ExpectedCoursePrice,ExpectedCurrency,DiscountRequested,LostReason,IsDuplicateCandidate
0,LEAD0000001,2024-01-01 00:00:01,2024-01-03 07:27:16,2024-01-01 00:27:16,NaN,NaN,artur.поляков1127@icloud.com,+380815156336,Ukraine,Kyiv Oblast,...,/ua/start,Mobile,prog.academy,referral,referral,0.0,UAH,False,NaN,False
1,LEAD0000002,2024-01-01 00:00:46,2024-01-04 10:40:33,2024-01-01 19:40:33,NaN,Katherine Сорокин,katherine.сорокин5692@gmail.com,+380113586793,Ukraine,Lviv Oblast,...,/ua/ai-automation,Mobile,NaN,direct,direct,21750.0,UAH,True,NaN,False
2,LEAD0000003,2024-01-01 00:09:30,01/01/2024,2024-01-01 08:56:19,NaN,Marie Małycha,marie.małycha4491@gmail.com,+380983031290,Ukraine,Kharkiv Oblast,...,/ua/qa,Desktop,t.me,telegram,telegram,28200.0,UAH,True,неправильный номер,False
3,LEAD0000004,2024-01-01 00:11:33,2024-01-02 22:56:50,2024-01-01 03:56:50,NaN,Tobiasz Брагина,tobiasz.брагина7657@gmail.com,+380674161203,Ukraine,Lviv Oblast,...,/ua/frontend,Desktop,t.me,telegram,telegram,28130.0,UAH,False,передумал,False
4,LEAD0000005,2024-01-01 00:21:07,2024-01-02 11:05:10,2024-01-01 11:05:10,NaN,Ruth Архипенко,ruth.архипенко894@outlook.com,+380659994537,Ukraine,Lviv Oblast,...,/ua/ai-automation,Desktop,prog.academy,referral,referral,38850.0,UAH,False,дорого,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51016,LEAD0051017D,2025-12-28 01:03:53,2025-12-28 19:10:39,2025-12-28 04:10:39,NaN,EMILY КУЗЬМИН,EMILY.КУЗЬМИН5334@UKR.NET,+380 81 019 2107,Ukraine,Kharkiv Oblast,...,/ua/start,Desktop,chatgpt.com,chatgpt.com,chatgpt.com,0.0,UAH,False,недозвон,True
51017,LEAD0051018D,2025-12-29 02:37:42,31.12.2025,2025-12-29 08:14:59,NaN,Добромысл Орлов,ДОБРОМЫСЛ.ОРЛОВ8428@ICLOUD.COM,+380 (30) 0527826,Ukraine,Odesa Oblast,...,/ua/start,Mobile,linkedin.com,linkedin,linkedin,0.0,UAH,False,дорого,True
51018,LEAD0051019D,12/29/2025,2025-12-31 03:47:29,2025-12-29 05:47:29,NaN,ПОРФИРИЙ БЕЗБОРОДЬКО,ПОРФИРИЙ.БЕЗБОРОДЬКО341@YAHOO.COM,380767275337,Ukraine,Lviv Oblast,...,/ua/python,Mobile,instagram.com,instagram,instagram,29280.0,UAH,True,перестал отвечать в чате,True
51019,LEAD0051020D,2025-12-29 12:26:56,2025-12-31 07:18:33,"Dec 29, 2025",NaN,Єва Лесик,Єва.Лесик9763@Gmail.Com,380-130-971-668,Ukraine,Odesa Oblast,...,/ua/qa,Mobile,NaN,direct,direct,31590.0,UAH,False,недозвон,True


In [26]:
is_unique_clean = out4_cleaned['LeadID'].is_unique
is_unique_clean


True

Checking for missing values

In [27]:
out4_cleaned.isnull().sum()

,0
LeadID,0
CreatedAt,0
UpdatedAt,0
FirstContactAt,3152
ConvertedAt,38254
FullName,1596
Email,0
Phone,1489
Country,0
Region,380


Датасет Leads характеризується високою якістю ключових полів, оскільки критичні ідентифікатори на кшталт LeadID та Email заповнені повністю. Пропуски в датах конвертації та першого контакту відображають природний стан воронки продажів, а не помилки в даних. Відсутні значення в маркетингових метках та географії легко усуваються заповненням категоріями 'not_set' та 'Unknown' без втрати рядків.

Data type and Handling missing values

In [28]:
out4_cleaned['LeadStatus'] = out4_cleaned['LeadStatus'].str.upper()
out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].str.upper()

/tmp/ipykernel_3387/1096259545.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['LeadStatus'] = out4_cleaned['LeadStatus'].str.upper()
/tmp/ipykernel_3387/1096259545.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].str.upper()


In [29]:
out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].replace({
     '-': 'UNKNOWN',
     '?': 'UNKNOWN',
     '--': 'UNKNOWN',
     'ТЕСТ': 'UNKNOWN'
})

/tmp/ipykernel_3387/717781697.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].replace({


In [30]:
out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].fillna('UNKNOWN')

/tmp/ipykernel_3387/3974181771.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['PreferredLanguage'] = out4_cleaned['PreferredLanguage'].fillna('UNKNOWN')


In [31]:
def clean_region(x):
    if pd.isna(x):
        return "unknown"
    x_str = str(x).strip()
    invalid_values = {"?", "--", "-", "тест", "unknown"}
    if x_str in invalid_values:
        return "unknown"
    return x_str

In [33]:
out4_cleaned['Region'] = out4_cleaned['Region'].apply(clean_region)

/tmp/ipykernel_3387/1145261236.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Region'] = out4_cleaned['Region'].apply(clean_region)


In [34]:
out4_cleaned['City'] = out4_cleaned['City'].str.strip().str.title()
unique_cities = out4_cleaned['City'].dropna().unique()

/tmp/ipykernel_3387/4075432411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['City'] = out4_cleaned['City'].str.strip().str.title()


In [35]:
!pip install fuzzywuzzy python-Levenshtein charset-normalizer

In [ ]:
out4_cleaned.dtypes

,0
LeadID,object
CreatedAt,object
UpdatedAt,object
FirstContactAt,object
ConvertedAt,object
FullName,object
Email,object
Phone,object
Country,object
Region,object


всі строки таблиці Leads мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як float/int, string, date, category, які добре працюють та займають менше оперативної пам'яті.

In [37]:
out4_cleaned['CourseID'] = out4_cleaned['CourseID'].astype(str)
out4_cleaned['CourseNameRaw'] = out4_cleaned['CourseNameRaw'].astype(str)
out4_cleaned['ManagerID'] = out4_cleaned['ManagerID'].astype(str)
out4_cleaned['FirstTouchSource'] = out4_cleaned['FirstTouchSource'].str.lower()
out4_cleaned['ReferrerDomain'] = out4_cleaned['ReferrerDomain'].astype(str).fillna('notset')
out4_cleaned['DeviceType'] = out4_cleaned['DeviceType'].str.upper()
out4_cleaned['Source'] = out4_cleaned['Source'].str.lower()
out4_cleaned['LeadScore'] = out4_cleaned['LeadScore'].astype('Int64').fillna(0)
out4_cleaned['LandingPage'] = out4_cleaned['LandingPage'].astype(str)
out4_cleaned['Content'] = out4_cleaned['Content'].astype(str).fillna('notset')
out4_cleaned['Medium'] = out4_cleaned['Medium'].astype(str)
out4_cleaned['Term'] = out4_cleaned['Term'].astype(str).fillna('notset')
out4_cleaned['LeadTemperature'] = out4_cleaned['LeadTemperature'].str.upper()
out4_cleaned['LeadID'] = out4_cleaned['LeadID'].astype(str)
out4_cleaned['FullName'] = out4_cleaned['FullName'].astype(str)
out4_cleaned['Campaign'].astype(str).fillna('notset')

/tmp/ipykernel_3387/3863048079.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['CourseID'] = out4_cleaned['CourseID'].astype(str)
/tmp/ipykernel_3387/3863048079.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['CourseNameRaw'] = out4_cleaned['CourseNameRaw'].astype(str)
/tmp/ipykernel_3387/3863048079.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the

,Campaign
0,nan
1,nan
2,Search - Курс Web Design - Ukraine New
3,da_free
4,nan
...,...
51016,search - brand - ukraine
51017,pmax_python_eu
51018,search - qa - eu
51019,nan


In [38]:
out4_cleaned['FirstTouchSource'] = out4_cleaned['FirstTouchSource'].replace({
  'www.google.com': 'google',
  'facebook_ads': 'facebook',
  'fb-insta': 'facebook',
  'fb': 'facebook',
  'www.google.com.ua': 'google'
})

/tmp/ipykernel_3387/577931672.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['FirstTouchSource'] = out4_cleaned['FirstTouchSource'].replace({


In [39]:
out4_cleaned['Source'] = out4_cleaned['Source'].replace({
 'www.google.com': 'google',
 'facebook_ads': 'facebook',
 'fb-insta': 'facebook',
 'fb': 'facebook',
 'meta': 'facebook',
  'www.google.com.ua': 'google'
})

/tmp/ipykernel_3387/3833480335.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Source'] = out4_cleaned['Source'].replace({


In [40]:
out4_cleaned['Medium'] = out4_cleaned['Medium'].replace({
    'none': 'notset'
})

/tmp/ipykernel_3387/3526644913.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Medium'] = out4_cleaned['Medium'].replace({


In [41]:
out4_cleaned['FullName_en'] = out4_cleaned['FullName'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)
out4_cleaned['FullName_en'] = out4_cleaned['FullName_en'].str.strip().str.title()

/tmp/ipykernel_3387/2549055616.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['FullName_en'] = out4_cleaned['FullName'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)
/tmp/ipykernel_3387/2549055616.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['FullName_en'] = out4_cleaned['FullName_en'].str.strip().str.title()


In [43]:
out4_cleaned['ManagerNameRaw_en'] = out4_cleaned['ManagerNameRaw'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)
out4_cleaned['ManagerNameRaw_en'] = out4_cleaned['ManagerNameRaw_en'].str.strip().str.title()

/tmp/ipykernel_3387/501754041.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['ManagerNameRaw_en'] = out4_cleaned['ManagerNameRaw'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)
/tmp/ipykernel_3387/501754041.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['ManagerNameRaw_en'] = out4_cleaned['ManagerNameRaw_en'].str.strip().str.title()


In [44]:
out4_cleaned['Email'] = out4_cleaned['Email'].astype(str).str.strip().str.lower()
out4_cleaned['Email_en'] = out4_cleaned['Email'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)

/tmp/ipykernel_3387/3943153192.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Email'] = out4_cleaned['Email'].astype(str).str.strip().str.lower()
/tmp/ipykernel_3387/3943153192.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Email_en'] = out4_cleaned['Email'].apply(lambda x: translit(str(x), 'ru', reversed=True) if pd.notna(x) else x)


In [45]:
out4_cleaned.drop(columns=['Email', 'FullName', 'ManagerNameRaw'], inplace=True)

/tmp/ipykernel_3387/1483320559.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned.drop(columns=['Email', 'FullName', 'ManagerNameRaw'], inplace=True)


In [46]:
def clean_phone(x):
    if pd.isna(x): return np.nan
    d = "".join(c for c in str(x) if c.isdigit())
    if d.startswith("380") and len(d) == 12: return "+" + d
    if d.startswith("0") and len(d) == 10: return "+38" + d
    return "+" + d if d else np.nan

In [47]:
out4_cleaned['Phone'] = out4_cleaned['Phone'].apply(clean_phone).fillna('notset')

/tmp/ipykernel_3387/3983760297.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Phone'] = out4_cleaned['Phone'].apply(clean_phone).fillna('notset')


In [48]:
out4_cleaned['Phone'] = out4_cleaned['Phone'].astype(str)
out4_cleaned['DiscountRequested'] = out4_cleaned['DiscountRequested'].astype(bool)
out4_cleaned['IsDuplicateCandidate'] = out4_cleaned['IsDuplicateCandidate'].astype(bool)
out4_cleaned['ExpectedCurrency'] = out4_cleaned['ExpectedCurrency'].astype(str)


/tmp/ipykernel_3387/2139181134.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Phone'] = out4_cleaned['Phone'].astype(str)
/tmp/ipykernel_3387/2139181134.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['DiscountRequested'] = out4_cleaned['DiscountRequested'].astype(bool)
/tmp/ipykernel_3387/2139181134.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in 

In [49]:
def clean_money(x):
    if pd.isna(x): return np.nan
    x = re.sub(r"[₴грн\s]", "", str(x)).replace(",", ".")
    x = re.sub(r"[^0-9.]", "", x)
    try: return float(x)
    except: return np.nan

In [50]:
out4_cleaned['ExpectedCoursePrice'] = out4_cleaned['ExpectedCoursePrice'].apply(clean_money)

/tmp/ipykernel_3387/31511190.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['ExpectedCoursePrice'] = out4_cleaned['ExpectedCoursePrice'].apply(clean_money)


In [51]:
out4_cleaned['Country'] = out4_cleaned['Country'].str.strip().str.title()
unique_countries = out4_cleaned['Country'].dropna().unique()

/tmp/ipykernel_3387/2794113003.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Country'] = out4_cleaned['Country'].str.strip().str.title()


In [52]:
from dateutil import parser

def parse_hybrid_date(val):
    if pd.isna(val): return pd.NaT
    try:
        return parser.parse(str(val), dayfirst=True)
    except:
        return pd.NaT

out4_cleaned['CreatedAt'] = out4_cleaned['CreatedAt'].apply(parse_hybrid_date)
out4_cleaned['UpdatedAt'] = out4_cleaned['UpdatedAt'].apply(parse_hybrid_date)
out4_cleaned['FirstContactAt'] = out4_cleaned['FirstContactAt'].apply(parse_hybrid_date)
out4_cleaned['ConvertedAt'] = out4_cleaned['ConvertedAt'].apply(parse_hybrid_date)

/tmp/ipykernel_3387/4137114331.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['CreatedAt'] = out4_cleaned['CreatedAt'].apply(parse_hybrid_date)
/tmp/ipykernel_3387/4137114331.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['UpdatedAt'] = out4_cleaned['UpdatedAt'].apply(parse_hybrid_date)
/tmp/ipykernel_3387/4137114331.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


Correction inconsistent data entries

In [53]:
!pip install fuzzywuzzy python-Levenshtein charset-normalizer

In [56]:
import numpy as np
import fuzzywuzzy
from fuzzywuzzy import process
import charset_normalizer

In [57]:
matches = fuzzywuzzy.process.extract("Ukraine", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches

[('Ukraine', 100),
 ('Ukraiine', 93),
 ('Ukraaine', 93),
 ('Ukkraine', 93),
 ('Ukrainne', 93),
 ('Ukrraine', 93),
 ('Ukrane', 92),
 ('Uraine', 92),
 ('Ukrine', 92),
 ('Ukaine', 92),
 ('Ukraie', 92),
 ('Ukra1Ne', 86),
 ('Ukriane', 86),
 ('Ukarine', 86),
 ('Ukranie', 86),
 ('Ukraien', 86),
 ('Urkaine', 86),
 ('Gerany', 46),
 ('Grmany', 46),
 ('Ua', 44)]

In [58]:
def replace_matches_in_column(df, column, string_to_match, min_ratio = 70):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")


In [59]:
replace_matches_in_column(
    df=out4_cleaned, column='Country', string_to_match='Ukraine')


All done!


In [60]:
matches_1 = fuzzywuzzy.process.extract("United Kingdom", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
replace_matches_in_column(df=out4_cleaned, column='Country', string_to_match='United Kingdom')

All done!


In [61]:
matches_2 = fuzzywuzzy.process.extract("Poland", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_2

[('Poland', 100),
 ('Pooland', 92),
 ('Polland', 92),
 ('Polaand', 92),
 ('Polannd', 92),
 ('Polnd', 91),
 ('Poand', 91),
 ('Polad', 91),
 ('Pland', 91),
 ('Poalnd', 83),
 ('Polnad', 83),
 ('P0Land', 83),
 ('Ploand', 83),
 ('Poladn', 83),
 ('Pl', 50),
 ('Pol@Nd', 50),
 ('Ukrane', 33),
 ('Uraine', 33),
 ('Gerany', 33),
 ('Ukaine', 33)]

In [62]:
def replace_matches_in_column_2(df, column, string_to_match, min_ratio = 40):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")


In [63]:
replace_matches_in_column_2(df=out4_cleaned, column='Country', string_to_match='Poland')

All done!


In [64]:
matches_3 = fuzzywuzzy.process.extract("Germany", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_3

[('Germany', 100),
 ('Germanny', 93),
 ('Germaany', 93),
 ('Gerrmany', 93),
 ('Gerany', 92),
 ('Germny', 92),
 ('Germay', 92),
 ('Grmany', 92),
 ('Geramny', 86),
 ('G3Rmany', 86),
 ('Germnay', 86),
 ('Gremany', 86),
 ('Germayn', 86),
 ('Ukrane', 46),
 ('Uraine', 46),
 ('Ukraine', 43),
 ('Ukra1Ne', 43),
 ('Ukriane', 43),
 ('Ukranie', 43),
 ('Ukraien', 43)]

In [65]:
replace_matches_in_column(df=out4_cleaned, column='Country', string_to_match='Germany')

All done!


In [66]:
matches_4 = fuzzywuzzy.process.extract("Czech Republic", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_4

[('Czech Republic', 100),
 ('Czechh Republic', 97),
 ('Czech Repuublic', 97),
 ('Czecch Republic', 97),
 ('Czech Republlic', 97),
 ('Czech Republiic', 97),
 ('Czech Rrepublic', 97),
 ('Czech Repblic', 96),
 ('Czech Rpublic', 96),
 ('Czechrepublic', 96),
 ('Czeh Republic', 96),
 ('Czech Epublic', 96),
 ('Czech Repulic', 96),
 ('Czch Republic', 96),
 ('Czech Repulbic', 93),
 ('Czech Republci', 93),
 ('Czech Repbulic', 93),
 ('Cz3Ch Republic', 93),
 ('Czech Republ1C', 93),
 ('Czech Reupblic', 93)]

In [67]:
replace_matches_in_column(df=out4_cleaned, column='Country', string_to_match='Czech Republic')

All done!


In [68]:
matches_5 = fuzzywuzzy.process.extract("Kazakhstan", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_5

[('Kazakhstan', 100),
 ('Kazakkhstan', 95),
 ('Kazakstan', 95),
 ('Kazakhsan', 95),
 ('Kazakhstaan', 95),
 ('Kaazakhstan', 95),
 ('Kazahstan', 95),
 ('Kazakhsttan', 95),
 ('Kaz@Khstan', 90),
 ('Kazakhstna', 90),
 ('Kzaakhstan', 90),
 ('Kazakhsatn', 90),
 ('Ukraaine', 44),
 ('Ukkraine', 44),
 ('Ukrane', 38),
 ('Ukaine', 38),
 ('United Stats', 36),
 ('Unite States', 36),
 ('Unied States', 36),
 ('Ukraine', 35)]

In [69]:
replace_matches_in_column(df=out4_cleaned, column='Country', string_to_match='Kazakhstan')

All done!


In [70]:
matches_6 = fuzzywuzzy.process.extract("United States", unique_countries, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_6

[('United States', 100),
 ('United $Tates', 96),
 ('United Stats', 96),
 ('Unite States', 96),
 ('Unied States', 96),
 ('Uniteed States', 96),
 ('Uniteds Tates', 92),
 ('United Stat3S', 92),
 ('Unietd States', 92),
 ('United St@Tes', 92),
 ('United Kigdom', 54),
 ('United Kindom', 54),
 ('United Kingdm', 54),
 ('United Kingdom', 52),
 ('United K1Ngdom', 52),
 ('United Kindgom', 52),
 ('United Kingodm', 52),
 ('United Kingdmo', 52),
 ('United Knigdom', 52),
 ('Uniteed Kingdom', 50)]

In [71]:
replace_matches_in_column(df=out4_cleaned, column='Country', string_to_match='United States')

All done!


In [72]:
out4_cleaned['Country'] = out4_cleaned['Country'].replace({
'Ua':'Ukraine',
'Kz':'Kazakhstan',
'Us': 'United States',
'De': 'Germany',
'Cz': 'Czech Republic',
'Ukr@Ine': 'Ukraine',
'Gb': 'United Kingdom'
}, regex=False)

/tmp/ipykernel_3387/2482907601.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['Country'] = out4_cleaned['Country'].replace({


In [73]:
out4_cleaned['City'] = out4_cleaned['City'].str.strip().str.title()
unique_cities = out4_cleaned['City'].dropna().unique()

/tmp/ipykernel_3387/4075432411.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['City'] = out4_cleaned['City'].str.strip().str.title()


In [74]:
matches_ = fuzzywuzzy.process.extract("Berlin", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_

[('Berlin', 100),
 ('Berliin', 92),
 ('Berrlin', 92),
 ('Beerlin', 92),
 ('Berllin', 92),
 ('Berln', 91),
 ('Belin', 91),
 ('Brlin', 91),
 ('Berin', 91),
 ('B3Rlin', 83),
 ('Beriln', 83),
 ('Brelin', 83),
 ('Berl1N', 83),
 ('Belrin', 83),
 ('Berlni', 83),
 ('Brno', 60),
 ('Bron', 60),
 ('Brnno', 55),
 ('Brrno', 55),
 ('Brnoo', 55)]

In [75]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Berlin')

All done!


In [76]:
matches_7 = fuzzywuzzy.process.extract("Chicago", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_7

[('Chicago', 100),
 ('Chiccago', 93),
 ('Chicaago', 93),
 ('Chhicago', 93),
 ('Chiicago', 93),
 ('Chcago', 92),
 ('Chiago', 92),
 ('Chicgo', 92),
 ('Cicago', 92),
 ('Chicao', 92),
 ('Chicaog', 86),
 ('Chic@Go', 86),
 ('Chicgao', 86),
 ('Ch1Cago', 86),
 ('Chciago', 86),
 ('Chiacgo', 86),
 ('Cihcago', 86),
 ('Hamurg', 46),
 ('Haburg', 46),
 ('Hambrg', 46)]

In [77]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Chicago')

All done!


In [78]:
matches_8 = fuzzywuzzy.process.extract("Odessa", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_8

[('Odessa', 100),
 ('Odesa', 91),
 ('Odeesa', 83),
 ('Oddesa', 83),
 ('Odea', 80),
 ('Odsa', 80),
 ('Oesa', 80),
 ('Desa', 80),
 ('Odsea', 73),
 ('Odeas', 73),
 ('Oedsa', 73),
 ('Od3Sa', 73),
 ('Ode$A', 55),
 ('Gdanssk', 46),
 ('Warssaw', 46),
 ('Asstana', 46),
 ('Vinnytssia', 38),
 ('Gdnsk', 36),
 ('Wasaw', 36),
 ('Gdank', 36)]

In [79]:
def replace_matches_in_column_3(df, column, string_to_match, min_ratio = 50):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")


In [80]:
replace_matches_in_column_3(df=out4_cleaned, column='City', string_to_match='Odessa')

All done!


In [81]:
matches_9 = fuzzywuzzy.process.extract("Lviv", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_9

[('Lviv', 100),
 ('Lviiv', 89),
 ('Lvviv', 89),
 ('Llviv', 89),
 ('Lvi V', 89),
 ('Lvv', 86),
 ('Liv', 86),
 ('Lvov', 75),
 ('Lv1V', 75),
 ('Lvvi', 75),
 ('Livv', 75),
 ('Vliv', 75),
 ('Kiv', 57),
 ('Yiv', 57),
 ('Kyiv', 50),
 ('Kiev', 50),
 ('Kyvi', 50),
 ('Kiyv', 50),
 ('Ykiv', 50),
 ('Kyyiv', 44)]

In [82]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Lviv')

All done!


In [83]:
matches_10 = fuzzywuzzy.process.extract("New York", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_10

[('New York', 100),
 ('New  York', 100),
 ('Neww York', 94),
 ('New Yyork', 94),
 ('Neew York', 94),
 ('New Yoork', 94),
 ('New Yorrk', 94),
 ('Nw York', 93),
 ('Newyork', 93),
 ('New Ork', 93),
 ('New Yok', 93),
 ('New Yrk', 93),
 ('N3W York', 88),
 ('Nwe York', 88),
 ('New Y0Rk', 88),
 ('Newy Ork', 88),
 ('New Yrok', 88),
 ('New Yokr', 88),
 ('Ne Wyork', 88),
 ('New Oyrk', 88)]

In [84]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='New York')

All done!


In [85]:
matches_11 = fuzzywuzzy.process.extract("Kharkiv", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_11

[('Kharkiv', 100),
 ('Khaarkiv', 93),
 ('Kharkiiv', 93),
 ('Kharrkiv', 93),
 ('Kharkkiv', 93),
 ('Khharkiv', 93),
 ('Khrkiv', 92),
 ('Khakiv', 92),
 ('Khariv', 92),
 ('Kharkv', 92),
 ('Karkiv', 92),
 ('Harkiv', 92),
 ('Kharkov', 86),
 ('Khark1V', 86),
 ('Khrakiv', 86),
 ('Kh@Rkiv', 86),
 ('Kahrkiv', 86),
 ('Kharkvi', 86),
 ('Kharikv', 86),
 ('Khakriv', 86)]

In [86]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Kharkiv')

All done!


In [87]:
matches_12 = fuzzywuzzy.process.extract("London", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_12

[('London', 100),
 ('Londoon', 92),
 ('Lonndon', 92),
 ('Londdon', 92),
 ('Loondon', 92),
 ('Lndon', 91),
 ('Lodon', 91),
 ('Lonon', 91),
 ('Ondon', 91),
 ('Lodnon', 83),
 ('Lonodn', 83),
 ('L0Ndon', 83),
 ('Londno', 83),
 ('Lond0N', 83),
 ('Unknown', 46),
 ('Bno', 44),
 ('Brno', 40),
 ('Lvov', 40),
 ('Odea', 40),
 ('Odsa', 40)]

In [88]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='London')

All done!


In [89]:
matches_13 = fuzzywuzzy.process.extract("Dnipro", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_13

[('Dnipro', 100),
 ('Dniprro', 92),
 ('Dnnipro', 92),
 ('Dniipro', 92),
 ('Dnippro', 92),
 ('Dipro', 91),
 ('Dnpro', 91),
 ('Dnipo', 91),
 ('Dniro', 91),
 ('Dnirpo', 83),
 ('Dn1Pro', 83),
 ('Dinpro', 83),
 ('Dnipor', 83),
 ('Dnpiro', 83),
 ('Bnro', 60),
 ('Lodnon', 50),
 ('Londno', 50),
 ('Bro', 44),
 ('Bno', 44),
 ('New Yrok', 43)]

In [90]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Dnipro')

All done!


In [91]:
matches_14 = fuzzywuzzy.process.extract("Prague", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_14

[('Prague', 100),
 ('Prrague', 92),
 ('Praague', 92),
 ('Pprague', 92),
 ('Praguue', 92),
 ('Pague', 91),
 ('Praue', 91),
 ('Prage', 91),
 ('Prageu', 83),
 ('Prauge', 83),
 ('Prgaue', 83),
 ('Pargue', 83),
 ('Praha', 55),
 ('Pr@Gue', 50),
 ('Kraow', 36),
 ('Ode$A', 36),
 ('Dipro', 36),
 ('Krakw', 36),
 ('Dnpro', 36),
 ('Austn', 36)]

In [92]:
replace_matches_in_column_2(df=out4_cleaned, column='City', string_to_match='Prague')

All done!


In [93]:
matches_15 = fuzzywuzzy.process.extract("Gdansk", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_15

[('Gdansk', 100),
 ('Gddansk', 92),
 ('Gdanssk', 92),
 ('Gdannsk', 92),
 ('Gdaansk', 92),
 ('Gdnsk', 91),
 ('Gansk', 91),
 ('Gdank', 91),
 ('Gdask', 91),
 ('Gdanks', 83),
 ('Gdnask', 83),
 ('Gadnsk', 83),
 ('Gdan$K', 83),
 ('Gd@Nsk', 83),
 ('Gdasnk', 83),
 ('Odeas', 55),
 ('Odea', 40),
 ('Odsa', 40),
 ('Manchster', 40),
 ('Mancester', 40)]

In [94]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Gdansk')

All done!


In [95]:
matches_16 = fuzzywuzzy.process.extract("Krakow", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_16

[('Krakow', 100),
 ('Krrakow', 92),
 ('Kraakow', 92),
 ('Krakoow', 92),
 ('Krakkow', 92),
 ('Kraow', 91),
 ('Krakw', 91),
 ('Kakow', 91),
 ('Krkow', 91),
 ('Krak0W', 83),
 ('Karkow', 83),
 ('Kraokw', 83),
 ('Krkaow', 83),
 ('Krakwo', 83),
 ('Kharkov', 62),
 ('Khrakiv', 62),
 ('Waraw', 55),
 ('Wrsaw', 55),
 ('Arsaw', 55),
 ('Warsaw', 50)]

In [96]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Krakow')

All done!


In [97]:
matches_17 = fuzzywuzzy.process.extract("Manchester", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_17

[('Manchester', 100),
 ('Mancheter', 95),
 ('Mannchester', 95),
 ('Manchesteer', 95),
 ('Manchster', 95),
 ('Mancester', 95),
 ('Mancheser', 95),
 ('Manchhester', 95),
 ('Manchestter', 95),
 ('Machester', 95),
 ('Mnchester', 95),
 ('Manchesster', 95),
 ('Manhester', 95),
 ('Mancheester', 95),
 ('Maanchester', 95),
 ('Manchseter', 90),
 ('Manchestre', 90),
 ('Manhcester', 90),
 ('Manchesetr', 90),
 ('M@Nchester', 90)]

In [98]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Manchester')

All done!


In [99]:
matches_18 = fuzzywuzzy.process.extract("Vinnytsia", unique_cities, limit=25, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_18

[('Vinnytsia', 100),
 ('Viinnytsia', 95),
 ('Vinnytsiia', 95),
 ('Vinnnytsia', 95),
 ('Vinnytssia', 95),
 ('Vinnyytsia', 95),
 ('Vinnyttsia', 95),
 ('Vinnytia', 94),
 ('Vinntsia', 94),
 ('Vinytsia', 94),
 ('Vinnytsa', 94),
 ('Vinnysia', 94),
 ('Vnnytsia', 94),
 ('V1Nnytsia', 89),
 ('Vinnyts1A', 89),
 ('Vinyntsia', 89),
 ('Vinntysia', 89),
 ('Vinnytisa', 89),
 ('Vninytsia', 89),
 ('Vinnytsai', 89),
 ('Vinnystia', 89),
 ('Vinnyt$Ia', 67),
 ('Lviiv', 43),
 ('Atsana', 40),
 ('Autsin', 40)]

In [100]:
replace_matches_in_column_3(df=out4_cleaned, column='City', string_to_match='Vinnytsia')

All done!


In [101]:
matches_19 = fuzzywuzzy.process.extract("Astana", unique_cities, limit=25, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_19

[('Astana', 100),
 ('Asttana', 92),
 ('Astaana', 92),
 ('Asstana', 92),
 ('Astanna', 92),
 ('Astna', 91),
 ('Asana', 91),
 ('Astaa', 91),
 ('Atsana', 83),
 ('Ast@Na', 83),
 ('Asatna', 83),
 ('A$Tana', 83),
 ('Astaan', 83),
 ('Astnaa', 83),
 ('Astin', 73),
 ('Austn', 73),
 ('Austin', 67),
 ('Ausitn', 67),
 ('Asutin', 67),
 ('Austni', 67),
 ('Aust1N', 67),
 ('Austiin', 62),
 ('Ausstin', 62),
 ('Austtin', 62),
 ('Auustin', 62)]

In [102]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match='Astana')

All done!


In [103]:
matches_20 = fuzzywuzzy.process.extract("Almaty", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_20

[('Almaty', 100),
 ('Allmaty', 92),
 ('Almaaty', 92),
 ('Almmaty', 92),
 ('Almatty', 92),
 ('Alaty', 91),
 ('Almay', 91),
 ('Amaty', 91),
 ('Almty', 91),
 ('Amlaty', 83),
 ('Alamty', 83),
 ('Almtay', 83),
 ('Almayt', 83),
 ('Alm@Ty', 83),
 ('Asatna', 50),
 ('Mancheter', 40),
 ('Manchster', 40),
 ('Mancester', 40),
 ('Machester', 40),
 ('Manhester', 40)]

In [104]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Almaty")

All done!


In [105]:
matches_21 = fuzzywuzzy.process.extract("Kyiv", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_21

[('Kyiv', 100),
 ('Kyyiv', 89),
 ('Kyiiv', 89),
 ('Kiv', 86),
 ('Kyv', 86),
 ('Yiv', 86),
 ('Kiev', 75),
 ('Kyvi', 75),
 ('Kiyv', 75),
 ('Ky1V', 75),
 ('Ykiv', 75),
 ('Khrkiv', 60),
 ('Khakiv', 60),
 ('Khariv', 60),
 ('Karkiv', 60),
 ('Harkiv', 60),
 ('Liv', 57),
 ('Kharkiv', 55),
 ('Khrakiv', 55),
 ('Kh@Rkiv', 55)]

In [106]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Kyiv")

All done!


In [107]:
matches_22 = fuzzywuzzy.process.extract("Munchen", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_22

[('München', 92),
 ('Munch', 83),
 ('Munich', 77),
 ('Muncih', 77),
 ('Mun1Ch', 77),
 ('Muinch', 77),
 ('Muunich', 71),
 ('Mmunich', 71),
 ('Municch', 71),
 ('Munnich', 71),
 ('Muniich', 71),
 ('Munih', 67),
 ('Muich', 67),
 ('Mnich', 67),
 ('Mancheter', 62),
 ('Manchster', 62),
 ('Mancheser', 62),
 ('Mnchester', 62),
 ('Munihc', 62),
 ('Mnuich', 62)]

In [108]:
def replace_matches_in_column_4(df, column, string_to_match, min_ratio = 63):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")


In [109]:
replace_matches_in_column_4(df=out4_cleaned, column='City', string_to_match="Munchen")

All done!


In [110]:
matches_23 = fuzzywuzzy.process.extract("Wroclaw", unique_cities, limit=30, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_23

[('Wroclaw', 100),
 ('Wrocclaw', 93),
 ('Wrroclaw', 93),
 ('Wrooclaw', 93),
 ('Wroclaaw', 93),
 ('Wrocllaw', 93),
 ('Wrocaw', 92),
 ('Woclaw', 92),
 ('Wrolaw', 92),
 ('Wroclw', 92),
 ('Wrclaw', 92),
 ('Wrocalw', 86),
 ('Wrcolaw', 86),
 ('Worclaw', 86),
 ('Wrolcaw', 86),
 ('Wroclwa', 86),
 ('Wr0Claw', 86),
 ('Wrocl@W', 71),
 ('Waraw', 67),
 ('Wrsaw', 67),
 ('Warsaw', 62),
 ('Wasraw', 62),
 ('Wrasaw', 62),
 ('Warasw', 62),
 ('Warrsaw', 57),
 ('Warssaw', 57),
 ('Warsaaw', 57),
 ('Waarsaw', 57),
 ('Warszawa', 53),
 ('Kraow', 50)]

In [112]:
def replace_matches_in_column_5(df, column, string_to_match, min_ratio = 70):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=None, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")


In [113]:
replace_matches_in_column_5(df=out4_cleaned, column='City', string_to_match="Wroclaw")

All done!


In [114]:
matches_24 = fuzzywuzzy.process.extract("Hamburg", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_24

[('Hamburg', 100),
 ('Hammburg', 93),
 ('Hamburrg', 93),
 ('Hambuurg', 93),
 ('Hambburg', 93),
 ('Haamburg', 93),
 ('Hamurg', 92),
 ('Haburg', 92),
 ('Hmburg', 92),
 ('Hambrg', 92),
 ('Hambug', 92),
 ('Hmaburg', 86),
 ('Hamubrg', 86),
 ('Hambrug', 86),
 ('Hambugr', 86),
 ('H@Mburg', 86),
 ('Habmurg', 86),
 ('Chcago', 46),
 ('Chiago', 46),
 ('Prauge', 46)]

In [115]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Hamburg")

All done!


In [116]:
matches_25 = fuzzywuzzy.process.extract("Warsawa", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_25

[('Warszawa', 93),
 ('Warsaw', 92),
 ('Warswa', 92),
 ('Warrsaw', 86),
 ('Warssaw', 86),
 ('Warsaaw', 86),
 ('Waarsaw', 86),
 ('Wasaw', 83),
 ('Waraw', 83),
 ('Wrsaw', 83),
 ('Arsaw', 83),
 ('Wasraw', 77),
 ('Wrasaw', 77),
 ('Warasw', 77),
 ('Asana', 67),
 ('Astaa', 67),
 ('Astana', 62),
 ('Wrocaw', 62),
 ('Atsana', 62),
 ('Wrolaw', 62)]

In [117]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Warsawa")

All done!


In [118]:
matches_26 = fuzzywuzzy.process.extract("Austin", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_26

[('Austin', 100),
 ('Austiin', 92),
 ('Ausstin', 92),
 ('Austtin', 92),
 ('Auustin', 92),
 ('Astin', 91),
 ('Austn', 91),
 ('Autin', 91),
 ('Ausin', 91),
 ('Ausitn', 83),
 ('Asutin', 83),
 ('Autsin', 83),
 ('Austni', 83),
 ('Aust1N', 83),
 ('Astna', 73),
 ('Astana', 67),
 ('Ast@Na', 67),
 ('Asatna', 67),
 ('Astaan', 67),
 ('Astnaa', 67)]

In [119]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Austin")

All done!


In [120]:
matches_27 = fuzzywuzzy.process.extract("Brno", unique_cities, limit=20, scorer=fuzzywuzzy.fuzz.token_sort_ratio)
matches_27

[('Brno', 100),
 ('Brnno', 89),
 ('Brrno', 89),
 ('Brnoo', 89),
 ('Bro', 86),
 ('Bno', 86),
 ('Bnro', 75),
 ('Bron', 75),
 ('Berln', 67),
 ('Brlin', 67),
 ('Berin', 67),
 ('Berlin', 60),
 ('B3Rlin', 60),
 ('Beriln', 60),
 ('Brelin', 60),
 ('Berl1N', 60),
 ('Belrin', 60),
 ('Berlni', 60),
 ('Berliin', 55),
 ('Berrlin', 55)]

In [121]:
replace_matches_in_column(df=out4_cleaned, column='City', string_to_match="Brno")

All done!


In [132]:
out4_cleaned['City'] = out4_cleaned['City'].replace({
    'Mnachester': 'Manchester',
    'Manch3Ster': 'Manchester',
    'Manchetser': 'Manchester',
    'Manchest3R': 'Manchester',
    'Wars@W': 'Warsawa',
    'W@Rsaw': 'Warsawa',
    'Kr@Kow': 'Krakow',
    'Munihc': 'Munchen',
    'Vinnyt$Ia': 'Vinnytsia',
    'Vinnystia': 'Vinnytsia',
    'Тест': 'Unknown',
    '?': 'Unknown',
    '-': 'Unknown',
    '--': 'Unknown',
    'Mnuich': 'Munchen'
})

/tmp/ipykernel_3387/1506558857.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned['City'] = out4_cleaned['City'].replace({


In [123]:
out4_cleaned

,LeadID,CreatedAt,UpdatedAt,FirstContactAt,ConvertedAt,Phone,Country,Region,City,PreferredLanguage,...,FirstTouchSource,LastTouchSource,ExpectedCoursePrice,ExpectedCurrency,DiscountRequested,LostReason,IsDuplicateCandidate,FullName_en,ManagerNameRaw_en,Email_en
0,LEAD0000001,2024-01-01 00:00:01,2024-03-01 07:27:16,2024-01-01 00:27:16,NaT,+380815156336,Ukraine,Kyiv Oblast,Unknown,RU,...,referral,referral,0.0,UAH,True,NaN,True,Nan,Vasil' G.,artur.poljakov1127@icloud.com
1,LEAD0000002,2024-01-01 00:00:46,2024-04-01 10:40:33,2024-01-01 19:40:33,NaT,+380113586793,Ukraine,Lviv Oblast,Lviv,UA,...,direct,direct,21750.0,UAH,True,NaN,True,Katherine Sorokin,John Z.,katherine.sorokin5692@gmail.com
2,LEAD0000003,2024-01-01 00:09:30,2024-01-01 00:00:00,2024-01-01 08:56:19,NaT,+380983031290,Ukraine,Kharkiv Oblast,Kharkiv,RU,...,telegram,telegram,28200.0,UAH,True,неправильный номер,True,Marie Małycha,Orest K.,marie.małycha4491@gmail.com
3,LEAD0000004,2024-01-01 00:11:33,2024-02-01 22:56:50,2024-01-01 03:56:50,NaT,+380674161203,Ukraine,Lviv Oblast,Lviv,RU,...,telegram,telegram,28130.0,UAH,True,передумал,True,Tobiasz Bragina,Oksana B,tobiasz.bragina7657@gmail.com
4,LEAD0000005,2024-01-01 00:21:07,2024-02-01 11:05:10,2024-01-01 11:05:10,NaT,+380659994537,Ukraine,Lviv Oblast,Lviv,UA,...,referral,referral,38850.0,UAH,True,дорого,True,Ruth Arhipenko,Ilarion D.,ruth.arhipenko894@outlook.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51016,LEAD0051017D,2025-12-28 01:03:53,2025-12-28 19:10:39,2025-12-28 04:10:39,NaT,+380810192107,Ukraine,Kharkiv Oblast,Kharkiv,UA,...,chatgpt.com,chatgpt.com,0.0,UAH,True,недозвон,True,Emily Kuz'Min,John Z.,emily.kuz'min5334@ukr.net
51017,LEAD0051018D,2025-12-29 02:37:42,2025-12-31 00:00:00,2025-12-29 08:14:59,NaT,+380300527826,Ukraine,Odesa Oblast,Odessa,RU,...,linkedin,linkedin,0.0,UAH,True,дорого,True,Dobromysl Orlov,Orest K.,dobromysl.orlov8428@icloud.com
51018,LEAD0051019D,2025-12-29 00:00:00,2025-12-31 03:47:29,2025-12-29 05:47:29,NaT,+380767275337,Ukraine,Lviv Oblast,Lviv,RU,...,instagram,instagram,29280.0,UAH,True,перестал отвечать в чате,True,Porfirij Bezborod'Ko,Vasyl F.,porfirij.bezborod'ko341@yahoo.com
51019,LEAD0051020D,2025-12-29 12:26:56,2025-12-31 07:18:33,2025-12-29 00:00:00,NaT,+380130971668,Ukraine,Odesa Oblast,Odessa,UA,...,direct,direct,31590.0,UAH,True,недозвон,True,Єva Lesik,John Z.,єva.lesik9763@gmail.com


In [124]:
out4_cleaned.drop(columns=['LostReason'], inplace=True)

/tmp/ipykernel_3387/3098147343.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out4_cleaned.drop(columns=['LostReason'], inplace=True)


In [133]:
out4_cleaned['City'].unique()

array(['Unknown', 'Lviv', 'Kharkiv', 'Berlin', 'Almaty', 'Vinnytsia',
       'Gdansk', 'Hamburg', nan, 'Odessa', 'New York', 'Prague', 'Dnipro',
       'Kyiv', 'Munchen', 'Krakow', 'Manchester', 'London', 'Wroclaw',
       'Warsawa', 'Austin', 'Brno', 'Chicago', 'Astana'], dtype=object)

In [126]:
out4_cleaned['Country'].unique()

array(['Ukraine', 'Germany', 'Kazakhstan', 'Poland', 'United States',
       'Czech Republic', 'United Kingdom'], dtype=object)

In [127]:
out4_cleaned['Region'].unique()

array(['Kyiv Oblast', 'Lviv Oblast', 'Kharkiv Oblast', 'Berlin Region',
       'Almaty Region', 'Vinnytsia Oblast', 'Gdansk Region',
       'Hamburg Region', 'Odesa Oblast', 'New York Region',
       'Dnipro Oblast', 'Prague Region', 'unknown', 'Krakow Region',
       'Manchester Region', 'London Region', 'Wroclaw Region',
       'Warsaw Region', 'Munich Region', 'Austin Region', 'Brno Region',
       'Chicago Region', 'Astana Region'], dtype=object)

In [128]:
out4_cleaned['PreferredLanguage'].unique()

array(['RU', 'UA', 'DE', 'EN', 'CS', 'PL', 'KK', 'UNKNOWN'], dtype=object)

In [134]:
# =====================================================
# EXPORT
# =====================================================
out4_cleaned.to_csv("Leads_Cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out4_cleaned.shape)

DONE: (51021, 34)


In [143]:
path5 = '/content/drive/MyDrive/Payments.csv'

In [144]:

# =====================================================
# LOAD Payments
# =====================================================
df5 = pd.read_csv(path5, encoding="utf-8-sig", dtype=str)
df5
out5 = df5.copy()
out5


,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,ExchangeRateToBase,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment
0,PAY0000001,LEAD0000061,STU000002,ENR000002,MGR0008,CRS0001,2024-01-15,Full,1,1,...,10.1015,37282.62,37282.62,Card,WayForPay,Canceled,NaN,NaN,INV-2024-000001,False
1,PAY0000002,LEAD0000087,STU000003,ENR000003,MGR0004,CRS0012,2024-01-04,Full,1,1,...,41.5325,32390.37,26820.86,Card,WayForPay,Completed,NaN,NaN,INV-2024-000002,False
2,PAY0000003,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-01-07,Installment,1,3,...,1.0,9300.0,7264.69,Card,WayForPay,Completed,NaN,NaN,INV-2024-000003,False
3,PAY0000004,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-02-09,Installment,2,3,...,1.0,9300.0,7303.64,Card,WayForPay,Completed,NaN,NaN,INV-2024-000004,False
4,PAY0000005,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-03-05,Installment,3,3,...,1.0,9300.0,7270.94,Card,WayForPay,Completed,NaN,NaN,INV-2024-000005,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17169,PAY0016138,LEAD0047076,STU011098,ENR011537,MGR0006,CRS0010,2025-12-13,Installment,1,2,...,1.0,14050.0,10305.47,Card,WayForPay,Completed,NaN,NaN,INV-2025-016138,False
17170,PAY0016382,LEAD0044829,STU011239,ENR011685,MGR0007,CRS0012,2025-12-12,Installment,1,2,...,1.0,16200.0,15871.38,Card,Wise,Completed,NaN,NaN,INV-2025-016382,False
17171,PAY0016484,LEAD0047670,STU011280,ENR011732,MGR0007,CRS0005,2026-02-15,Installment,3,3,...,1.0,9533.33,6949.07,Wise,Wise,Completed,NaN,NaN,INV-2026-016484,False
17172,PAY0016748,LEAD0047494,STU011445,ENR011909,MGR0004,CRS0006,2026-04-30,Installment,5,6,...,1.0,6350.0,6163.91,Card,PrivatBank,Completed,NaN,NaN,INV-2026-016748,False


Data Audit

In [145]:
out5.shape

(17174, 26)

PaymentID - Primary Key,  and Foreign key - LeadID,	StudentID,EnrollmentID,	ManagerID,	CourseID.

In [146]:
is_unique5 = out5['PaymentID'].is_unique
print(is_unique5)



False


In [147]:
total_duplicates5 = out5.duplicated().sum()
total_duplicates5


np.int64(74)

In [148]:
out5_cleaned = out5.drop_duplicates(subset=['PaymentID'], keep='first')
out5_cleaned

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,ExchangeRateToBase,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment
0,PAY0000001,LEAD0000061,STU000002,ENR000002,MGR0008,CRS0001,2024-01-15,Full,1,1,...,10.1015,37282.62,37282.62,Card,WayForPay,Canceled,NaN,NaN,INV-2024-000001,False
1,PAY0000002,LEAD0000087,STU000003,ENR000003,MGR0004,CRS0012,2024-01-04,Full,1,1,...,41.5325,32390.37,26820.86,Card,WayForPay,Completed,NaN,NaN,INV-2024-000002,False
2,PAY0000003,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-01-07,Installment,1,3,...,1.0,9300.0,7264.69,Card,WayForPay,Completed,NaN,NaN,INV-2024-000003,False
3,PAY0000004,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-02-09,Installment,2,3,...,1.0,9300.0,7303.64,Card,WayForPay,Completed,NaN,NaN,INV-2024-000004,False
4,PAY0000005,LEAD0000301,STU000005,ENR000005,MGR0007,CRS0017,2024-03-05,Installment,3,3,...,1.0,9300.0,7270.94,Card,WayForPay,Completed,NaN,NaN,INV-2024-000005,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17095,PAY0017096,LEAD0046995,STU011675,ENR012160,MGR0003,CRS0005,2026-03-16,Installment,3,6,...,1.0,4766.67,3336.67,Wise,PrivatBank,Failed,NaN,NaN,INV-2026-017096,False
17096,PAY0017097,LEAD0046995,STU011675,ENR012160,MGR0003,CRS0005,2026-04-16,Installment,4,6,...,1.0,4766.67,3336.67,Wise,PrivatBank,Failed,NaN,NaN,INV-2026-017097,False
17097,PAY0017098,LEAD0046995,STU011675,ENR012160,MGR0003,CRS0005,2026-05-13,Installment,5,6,...,1.0,4766.67,3336.67,Wise,PrivatBank,Failed,NaN,NaN,INV-2026-017098,False
17098,PAY0017099,LEAD0046995,STU011675,ENR012160,MGR0003,CRS0005,2026-06-12,Installment,6,6,...,1.0,4766.67,3336.67,Wise,PrivatBank,Failed,NaN,NaN,INV-2026-017099,False


In [149]:
out5_cleaned['PaymentID'].is_unique

True

Checking for missing values

In [150]:
out5_cleaned.isnull().sum()

,0
PaymentID,0
LeadID,0
StudentID,0
EnrollmentID,0
ManagerID,0
CourseID,0
PaymentDate,75
PaymentType,0
InstallmentNumber,0
InstallmentCount,0


В стовпці payment date 75 пропущених значень, в стовпці refund date і refund reason 16168 пропущених значень. Це означає, що в даних випадках не було повернення грошей, не було возврату. Тому пропонуємо створити бінарний флаг повернення через створення стовпця is refunded.



Part 2 Data Cleaning

Data Types and Dates

In [151]:
out5_cleaned['PaymentID'] = out5_cleaned['PaymentID'].astype(str)
out5_cleaned['LeadID'] = out5_cleaned['LeadID'].astype(str)
out5_cleaned['StudentID'] = out5_cleaned['StudentID'].astype(str)
out5_cleaned['EnrollmentID'] = out5_cleaned['EnrollmentID'].astype(str)
out5_cleaned['ManagerID'] = out5_cleaned['ManagerID'].astype(str)
out5_cleaned['CourseID'] = out5_cleaned['CourseID'].astype(str)
out5_cleaned['PaymentType'] = out5_cleaned['PaymentType'].astype(str)
out5_cleaned['InstallmentNumber'] = out5_cleaned['InstallmentNumber'].astype(int)
out5_cleaned['InstallmentCount'] = out5_cleaned['InstallmentCount'].astype(int)
out5_cleaned['IsTestPayment'] = out5_cleaned['IsTestPayment'].astype(bool)
out5_cleaned['PaymentMethod'] = out5_cleaned['PaymentMethod'].astype(str)
out5_cleaned['PaymentProvider'] = out5_cleaned['PaymentProvider'].astype(str)
out5_cleaned['PaymentStatus'] = out5_cleaned['PaymentStatus'].astype(str)
out5_cleaned['RefundReason'] = out5_cleaned['RefundReason'].astype(str)
out5_cleaned['InvoiceNumber'] = out5_cleaned['InvoiceNumber'].astype(str)

/tmp/ipykernel_3387/918692688.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['PaymentID'] = out5_cleaned['PaymentID'].astype(str)
/tmp/ipykernel_3387/918692688.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['LeadID'] = out5_cleaned['LeadID'].astype(str)
/tmp/ipykernel_3387/918692688.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation:

Using function clean_money

In [152]:
out5_cleaned['GrossAmount'] = out5_cleaned['GrossAmount'].apply(clean_money)
out5_cleaned['DiscountAmount'] = out5_cleaned['DiscountAmount'].apply(clean_money)
out5_cleaned['RefundAmount'] = out5_cleaned['RefundAmount'].apply(clean_money)
out5_cleaned['ProcessingFee'] = out5_cleaned['ProcessingFee'].apply(clean_money)
out5_cleaned['NetAmount'] = out5_cleaned['NetAmount'].apply(clean_money)
out5_cleaned['ExchangeRateToBase'] = out5_cleaned['ExchangeRateToBase'].apply(clean_money)
out5_cleaned['GrossAmountBase'] = out5_cleaned['GrossAmountBase'].apply(clean_money)
out5_cleaned['NetAmountBase'] = out5_cleaned['NetAmountBase'].apply(clean_money)

/tmp/ipykernel_3387/1784587514.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['GrossAmount'] = out5_cleaned['GrossAmount'].apply(clean_money)
/tmp/ipykernel_3387/1784587514.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['DiscountAmount'] = out5_cleaned['DiscountAmount'].apply(clean_money)
/tmp/ipykernel_3387/1784587514.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



In [153]:
out5_cleaned['Currency'] = out5_cleaned['Currency'].str.upper()
out5_cleaned['Currency'] = out5_cleaned['Currency'].replace({
  'ГРН':'UAH',
  'EURO':'EUR',
  'DOLLAR':'USD'
})

/tmp/ipykernel_3387/493695462.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['Currency'] = out5_cleaned['Currency'].str.upper()
/tmp/ipykernel_3387/493695462.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['Currency'] = out5_cleaned['Currency'].replace({


In [155]:

out5_cleaned['InvoiceNumber'] = out5_cleaned['InvoiceNumber'].astype(str)

/tmp/ipykernel_3387/2845195216.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['InvoiceNumber'] = out5_cleaned['InvoiceNumber'].astype(str)


In [159]:

out5_cleaned['RefundDate'] = out5_cleaned['RefundDate'].apply(parse_hybrid_date)
out5_cleaned['PaymentDate'] = out5_cleaned['PaymentDate'].apply(parse_hybrid_date)


/tmp/ipykernel_3387/4169689857.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['RefundDate'] = out5_cleaned['RefundDate'].apply(parse_hybrid_date)
/tmp/ipykernel_3387/4169689857.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['PaymentDate'] = out5_cleaned['PaymentDate'].apply(parse_hybrid_date)


Handling missing values

In [157]:
out5_cleaned['is_refunded'] = out5_cleaned['RefundDate'].notna().astype(int)

/tmp/ipykernel_3387/3113833788.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['is_refunded'] = out5_cleaned['RefundDate'].notna().astype(int)


In [161]:
out5_cleaned['PaymentDate'] = out5_cleaned['PaymentDate'].fillna(out5_cleaned['RefundDate'])

/tmp/ipykernel_3387/2270621612.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out5_cleaned['PaymentDate'] = out5_cleaned['PaymentDate'].fillna(out5_cleaned['RefundDate'])


In [160]:
invalid_dates = out5_cleaned[out5_cleaned['RefundDate'] < out5_cleaned['PaymentDate']]
invalid_dates

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment,is_refunded


In [163]:
invalid_number = out5_cleaned[out5_cleaned['InstallmentNumber'] > out5_cleaned['InstallmentCount']]
invalid_number

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment,is_refunded
14,PAY0000015,LEAD0000662,STU000020,ENR000020,MGR0007,CRS0001,2024-01-18,Full,2,1,...,37600.00,25706.01,Wise,Monobank,Completed,NaT,nan,INV-2024-000015,True,0
352,PAY0000353,LEAD0000193,STU000278,ENR000278,MGR0002,CRS0011,2024-02-14,Full,2,1,...,27571.83,22770.43,Wise,WayForPay,Completed,NaT,nan,INV-2024-000353,True,0
465,PAY0000466,LEAD0000434,STU000370,ENR000370,MGR0007,CRS0002,2024-02-07,Installment,8,6,...,6025.82,5883.16,Card,PrivatBank,Completed,NaT,nan,INV-2024-000466,True,0
494,PAY0000495,LEAD0000641,STU000391,ENR000391,MGR0002,CRS0007,2024-02-15,Installment,5,3,...,13912.18,13015.18,Wise,Wise,Completed,NaT,nan,INV-2024-000495,True,0
604,PAY0000605,LEAD0001685,STU000464,ENR000464,MGR0002,CRS0006,2024-04-25,Installment,7,6,...,6350.00,4934.44,Wise,Wise,Completed,NaT,nan,INV-2024-000605,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16271,PAY0016272,LEAD0045335,STU011183,ENR011627,MGR0003,CRS0006,2025-12-18,Installment,8,6,...,6242.89,5235.18,Card,WayForPay,Completed,NaT,nan,INV 2025 016272,True,0
16586,PAY0016587,LEAD0046956,STU011339,ENR011799,MGR0007,CRS0004,2026-03-24,Installment,5,4,...,2175.00,1107.25,Card,USDT,Partially Refunded,2026-04-05,Course mismatch,INV-2026-016587,True,1
16690,PAY0016691,LEAD0047386,STU011417,ENR011880,MGR0004,CRS0017,2025-12-31,Full,2,1,...,27900.00,25893.17,Card,USDT,Completed,NaT,nan,INV-2025-016691,True,0
16937,PAY0016938,LEAD0046864,STU011569,ENR012044,MGR0006,CRS0003,2026-01-11,Installment,5,3,...,17466.67,17145.86,Card,PrivatBank,Completed,NaT,nan,INV-2026-016938,True,0


In [164]:
invalid_gross_amount = out5_cleaned[out5_cleaned['GrossAmount'] < 0]
invalid_gross_amount

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment,is_refunded


In [165]:
invalid_refund_amount = out5_cleaned[out5_cleaned['GrossAmount'] < out5_cleaned['RefundAmount']]
invalid_refund_amount

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment,is_refunded
234,PAY0000235,LEAD0001646,STU000183,ENR000183,MGR0002,CRS0002,2024-04-30,Installment,4,4,...,9200.00,9052.57,Crypto,Wise,Refunded,NaT,nan,INV-2024-000235,True,0
394,PAY0000395,LEAD0001581,STU000323,ENR000323,MGR0003,CRS0005,2024-02-11,Full,1,1,...,28600.00,628.06,Card,Wise,Refunded,2024-02-23,Financial reasons,INV-2024-000395,True,1
498,PAY0000499,LEAD0001334,STU000393,ENR000393,MGR0002,CRS0014,2024-02-20,Full,1,1,...,21796.90,21379.25,Card,Wise,Refunded,NaT,nan,INV-2024-000499,True,0
639,PAY0000640,LEAD0001453,STU000486,ENR000486,MGR0005,CRS0014,2024-02-21,Installment,1,6,...,3600.00,2645.28,Wise,USDT,Refunded,NaT,nan,INV-2024-000640,True,0
944,PAY0000945,LEAD0003626,STU000766,ENR000768,MGR0002,CRS0012,2024-02-27,Full,1,1,...,32448.37,31548.84,Card,Monobank,Refunded,NaT,nan,INV-2024-000945,True,0
1490,PAY0001491,LEAD0003173,STU001200,ENR001206,MGR0007,CRS0014,2024-07-25,Installment,5,6,...,3600.00,3377.86,BankTransfer,Monobank,Refunded,NaT,nan,INV-2024-001491,True,0
1618,PAY0001619,LEAD0005765,STU001310,ENR001316,MGR0003,CRS0017,2024-05-23,Installment,3,3,...,9289.95,6967.23,Wise,Wise,Refunded,NaT,nan,INV-2024-001619,True,0
1645,PAY0001646,LEAD0005279,STU001324,ENR001330,MGR0002,CRS0005,2024-06-07,Installment,3,6,...,5069.95,5069.95,Card,USDT,Refunded,NaT,nan,INV-2024-001646,True,0
1912,PAY0001913,LEAD0005855,STU001550,ENR001557,MGR0002,CRS0012,2024-05-15,Installment,2,2,...,16200.00,9083.52,BankTransfer,Monobank,Refunded,2024-06-09,Technical issue,INV-2024-001913,True,1
2163,PAY0002164,LEAD0005677,STU001723,ENR001731,MGR0002,CRS0005,2024-04-14,Full,1,1,...,28854.66,9717.90,Wise,Wise,Refunded,2024-06-17,Changed mind,INV-2024-002164,True,1


In [166]:
invalid_net_amount = out5_cleaned[out5_cleaned['NetAmount'] < 0]
invalid_net_amount

,PaymentID,LeadID,StudentID,EnrollmentID,ManagerID,CourseID,PaymentDate,PaymentType,InstallmentNumber,InstallmentCount,...,GrossAmountBase,NetAmountBase,PaymentMethod,PaymentProvider,PaymentStatus,RefundDate,RefundReason,InvoiceNumber,IsTestPayment,is_refunded


In [167]:
# =====================================================
# EXPORT
# =====================================================
out5_cleaned.to_csv("Payments_Cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out5_cleaned.shape)

DONE: (17100, 27)


In [ ]:
path6 = '/content/drive/MyDrive/MarketingSpend.csv'

In [ ]:
# =====================================================
# LOAD MarketingSpend
# =====================================================
df6 = pd.read_csv(path6, encoding="utf-8-sig", dtype=str)
df6
out6 = df6.copy()
out6


,Date,Source,Medium,Campaign,Country,DeviceType,Impressions,Clicks,Sessions,LeadsReported,Spend,Currency,SpendBaseCurrency,PlatformConversions,CampaignObjective,CampaignStatus
0,2024-01-01,google,cpc,Search - QA - Ukraine,Ukraine,Desktop,6309,241,218,6,0.0,UAH,0.0,1,Conversions,Paused
1,2024-01-02,google,cpc,Search - QA - Ukraine,Ukraine,Mobile,10145,494,429,12,0.0,UAH,0.0,2,Conversions,Paused
2,2024-01-03,google,cpc,Search - QA - Ukraine,Ukraine,Desktop,1555,59,45,0,0.0,UAH,0.0,0,Conversions,Paused
3,2024-01-04,google,cpc,Search - QA - Ukraine,Ukraine,Tablet,7460,179,136,2,0.0,UAH,0.0,0,Conversions,Paused
4,2024-01-05,google,cpc,Search - QA - Ukraine,Ukraine,Mobile,5900,325,314,7,0.0,UAH,0.0,1,Conversions,Paused
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13963,2025-04-13,referral,referral,NaN,United States,Mobile,1955,51,49,3,0.0,UAH,0.0,0,Referral,Active
13964,2025-11-23,referral,referral,NaN,Ukraine,Mobile,1756,94,91,3,0.0,UAH,0.0,1,Referral,Active
13965,2024-01-27,direct,direct,NaN,Ukraine,Mobile,682,50,41,2,0.0,UAH,0.0,0,Direct,Active
13966,2025-07-19,direct,direct,NaN,Ukraine,Desktop,1359,90,84,3,0.0,UAH,0.0,0,Direct,Active


Data Audit

In [ ]:
out6.shape

(13968, 16)

Checking for missing values

In [ ]:
out6.isnull().sum()


,0
Date,0
Source,0
Medium,0
Campaign,2938
Country,0
DeviceType,0
Impressions,0
Clicks,0
Sessions,0
LeadsReported,0


пропущені значення є тільки в стовбці Campaign, Spend. Пропущені значення в стовбці Campaign є сенс замінити на "not set", а пропущені значення в стовбці spend на 0.

In [ ]:
out6.dtypes

,0
Date,object
Source,object
Medium,object
Campaign,object
Country,object
DeviceType,object
Impressions,object
Clicks,object
Sessions,object
LeadsReported,object


всі строки таблиці MarketingSpend мають змішаний тип даних object. Для подальших дій (математичних тощо) необхідно перетворити цей тип на більш вживані типи такі як int, string, date, number, які добре працюють та займають менше оперативної пам'яті

Checking for duplicates

In [ ]:
total_duplicates6 = out6.duplicated().sum()
total_duplicates6


np.int64(79)

In [ ]:
out6_cleaned = out6.drop_duplicates()


Part 2 Data Cleaning

Data Types and Dates


In [ ]:
out6_cleaned['Source'] = out6_cleaned['Source'].astype(str)
out6_cleaned['Medium']  = out6_cleaned['Medium'].astype(str)
out6_cleaned['Campaign']  = out6_cleaned['Campaign'].astype(str)
out6_cleaned['Country']  = out6_cleaned['Country'].astype(str)
out6_cleaned['Country']  = out6_cleaned['Country'].astype(str)
out6_cleaned['DeviceType']  = out6_cleaned['DeviceType'].astype(str)
out6_cleaned['Impressions']  = out6_cleaned['Impressions'].astype(int)
out6_cleaned['Clicks']  = out6_cleaned['Clicks'].astype(int)
out6_cleaned['Sessions']  = out6_cleaned['Sessions'].astype(int)
out6_cleaned['LeadsReported']  = out6_cleaned['LeadsReported'].astype(int)
out6_cleaned['Currency']  = out6_cleaned['Currency'].astype(str)
out6_cleaned['PlatformConversions']  = out6_cleaned['PlatformConversions'].astype(int)
out6_cleaned['CampaignObjective']  = out6_cleaned['CampaignObjective'].astype(str)
out6_cleaned['CampaignStatus']  = out6_cleaned['CampaignStatus'].astype(str)




/tmp/ipykernel_3718/2358688655.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Source'] = out6_cleaned['Source'].astype(str)
/tmp/ipykernel_3718/2358688655.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Medium']  = out6_cleaned['Medium'].astype(str)
/tmp/ipykernel_3718/2358688655.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: h

In [ ]:
out6_cleaned['Spend'] = out6_cleaned['Spend'].apply(clean_money)
out6_cleaned['SpendBaseCurrency'] = out6_cleaned['SpendBaseCurrency'].apply(clean_money)

/tmp/ipykernel_3718/2981696327.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Spend'] = out6_cleaned['Spend'].apply(clean_money)
/tmp/ipykernel_3718/2981696327.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['SpendBaseCurrency'] = out6_cleaned['SpendBaseCurrency'].apply(clean_money)


In [ ]:
out6_cleaned['Date'] = out6_cleaned['Date'].apply(parse_hybrid_date)

/tmp/ipykernel_3718/50724491.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Date'] = out6_cleaned['Date'].apply(parse_hybrid_date)


Handling missing values

In [ ]:
out6_cleaned['Campaign'] = out6_cleaned['Campaign'].fillna('notset')
out6_cleaned['Spend'] = out6_cleaned['Spend'].fillna(0)

/tmp/ipykernel_3718/3710625160.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Campaign'] = out6_cleaned['Campaign'].fillna('notset')
/tmp/ipykernel_3718/3710625160.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Spend'] = out6_cleaned['Spend'].fillna(0)


Correction inconsistent entries

In [ ]:
out6_cleaned['Source'].unique()
out6_cleaned['Source'] = out6_cleaned['Source'].replace({
  'meta': 'facebook',
  'fb-insta': 'facebook'
})

/tmp/ipykernel_3718/3943472680.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out6_cleaned['Source'] = out6_cleaned['Source'].replace({


In [ ]:
# =====================================================
# EXPORT
# =====================================================
out6_cleaned.to_csv("MarketingSpend_cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out6_cleaned.shape)

DONE: (13889, 16)


In [ ]:
path7 = '/content/drive/MyDrive/StudentActivity.csv'

In [ ]:
# =====================================================
# LOAD StudentActivity
# =====================================================
df7 = pd.read_csv(path7, encoding="utf-8-sig", dtype=str)
df7
out7 = df7.copy()
out7

,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
0,ACT00000001,2024-01-14,STU000001,ENR000001,CRS0008,7,7,3,2,3,2,1,4,59.3,175,3,3,2,6.4,2024-01-20 10:52:47
1,ACT00000002,2024-01-21,STU000001,ENR000001,CRS0008,5,2,3,2,3,2,1,2,43.5,43,0,0,0,5.6,2024-01-27 08:40:04
2,ACT00000003,2024-01-28,STU000001,ENR000001,CRS0008,0,0,0,0,1,0,0,0,0.0,0,0,1,0,0.0,2024-01-30 19:28:50
3,ACT00000004,2024-01-17,STU000002,ENR000002,CRS0001,0,3,3,2,2,1,1,2,74.4,107,0,1,2,4.6,2024-01-18 03:11:31
4,ACT00000005,2024-01-24,STU000002,ENR000002,CRS0001,4,0,3,3,2,1,1,3,61.6,57,0,0,0,2.4,2024-01-29 05:17:54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111636,ACT00112016,2025-12-03,STU011655,ENR012139,CRS0014,2,2,1,1,1,0,0,1,28.1,0,0,0,0,0.8,2025-12-04 17:06:53
111637,ACT00112019,2025-12-24,STU011655,ENR012139,CRS0014,0,0,0,0,2,0,0,1,41.1,2,0,3,1,0.6,2025-12-25 11:14:05
111638,ACT00112057,2025-12-03,STU011664,ENR012148,CRS0014,10,7,3,2,2,2,2,4,45.6,218,2,3,4,9.0,2025-12-09 04:16:48
111639,ACT00112087,2025-12-03,STU011671,ENR012155,CRS0010,8,7,4,3,3,2,1,2,85.6,241,1,3,2,2.9,2025-12-08 16:21:07


Data Audit

In [ ]:
out7.shape

(111641, 20)

StudentID - Primary Key and EnrollmentID,CourseID - Foreign key.

In [ ]:
is_unique7 = out7['StudentID'].is_unique
print(is_unique7)


False


Checking for duplicates

In [ ]:
total_duplicates7 = out7.duplicated().sum()
total_duplicates7


np.int64(567)

In [ ]:
out7_cleaned = out7.drop_duplicates()
out7_cleaned


,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
0,ACT00000001,2024-01-14,STU000001,ENR000001,CRS0008,7,7,3,2,3,2,1,4,59.3,175,3,3,2,6.4,2024-01-20 10:52:47
1,ACT00000002,2024-01-21,STU000001,ENR000001,CRS0008,5,2,3,2,3,2,1,2,43.5,43,0,0,0,5.6,2024-01-27 08:40:04
2,ACT00000003,2024-01-28,STU000001,ENR000001,CRS0008,0,0,0,0,1,0,0,0,0.0,0,0,1,0,0.0,2024-01-30 19:28:50
3,ACT00000004,2024-01-17,STU000002,ENR000002,CRS0001,0,3,3,2,2,1,1,2,74.4,107,0,1,2,4.6,2024-01-18 03:11:31
4,ACT00000005,2024-01-24,STU000002,ENR000002,CRS0001,4,0,3,3,2,1,1,3,61.6,57,0,0,0,2.4,2024-01-29 05:17:54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111328,ACT00052361,2024-12-25,STU005019,ENR005133,CRS0012,8,7,2,2,3,1,1,5,82.6,269,1,1,4,4.4,2024-12-27 17:32:05
111468,ACT00080531,2025-05-07,STU007541,ENR007762,CRS0014,3,0,0,0,3,1,1,0,0.0,33,0,0,0,1.1,2025-05-08 04:58:03
111523,ACT00090465,2025-06-01,STU008399,ENR008670,CRS0008,6,6,5,4,3,3,2,4,65.2,179,1,0,4,5.4,2025-06-07 10:01:03
111537,ACT00092541,2025-08-28,STU008603,ENR008884,CRS0016,8,6,2,1,2,1,1,3,45.0,116,0,2,2,2.9,2025-08-29 05:19:19


In [ ]:
uniq = out7_cleaned['ActivityID'].is_unique
uniq

True

Checking for missing values

In [ ]:
out7_cleaned.isnull().sum()

,0
ActivityID,0
ActivityWeek,0
StudentID,0
EnrollmentID,0
CourseID,0
Logins,0
ActiveDays,0
LessonsViewed,0
LessonsCompleted,0
HomeworkAssigned,0


В таблиці StudentActivity немає missing values

In [ ]:
out7_cleaned.dtypes

,0
ActivityID,object
ActivityWeek,object
StudentID,object
EnrollmentID,object
CourseID,object
Logins,object
ActiveDays,object
LessonsViewed,object
LessonsCompleted,object
HomeworkAssigned,object


Part 2 Data Cleaning

Data Types and Dates

In [ ]:
out7_cleaned['ActivityID'] = out7_cleaned['ActivityID'].astype(str)
out7_cleaned['StudentID']  = out7_cleaned['StudentID'].astype(str)
out7_cleaned['EnrollmentID']  = out7_cleaned['EnrollmentID'].astype(str)
out7_cleaned['CourseID']  = out7_cleaned['CourseID'].astype(str)
out7_cleaned['Logins']  = out7_cleaned['Logins'].astype(int)
out7_cleaned['ActiveDays']  = out7_cleaned['ActiveDays'].astype(int)
out7_cleaned['LessonsViewed']  = out7_cleaned['LessonsViewed'].astype(int)
out7_cleaned['LessonsCompleted']  = out7_cleaned['LessonsCompleted'].astype(int)
out7_cleaned['HomeworkAssigned']  = out7_cleaned['HomeworkAssigned'].astype(int)
out7_cleaned['HomeworkSubmitted']  = out7_cleaned['HomeworkSubmitted'].astype(int)
out7_cleaned['HomeworkAccepted']  = out7_cleaned['HomeworkAccepted'].astype(int)
out7_cleaned['QuizAttempts']  = out7_cleaned['QuizAttempts'].astype(int)
out7_cleaned['AverageQuizScore']  = out7_cleaned['AverageQuizScore'].astype(float)
out7_cleaned['VideoMinutesWatched']  = out7_cleaned['VideoMinutesWatched'].astype(int)
out7_cleaned['LiveLessonsAttended']  = out7_cleaned['LiveLessonsAttended'].astype(int)
out7_cleaned['QuestionsAsked']  = out7_cleaned['QuestionsAsked'].astype(int)
out7_cleaned['MentorMessages']  = out7_cleaned['MentorMessages'].astype(int)
out7_cleaned['PlatformHours']  = out7_cleaned['PlatformHours'].astype(float)



/tmp/ipykernel_3718/1287011462.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out7_cleaned['ActivityID'] = out7_cleaned['ActivityID'].astype(str)
/tmp/ipykernel_3718/1287011462.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out7_cleaned['StudentID']  = out7_cleaned['StudentID'].astype(str)
/tmp/ipykernel_3718/1287011462.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the do

In [ ]:
out7_cleaned['ActivityWeek'] = out7_cleaned['ActivityWeek'].apply(parse_hybrid_date)
out7_cleaned['LastActivityAt'] = out7_cleaned['LastActivityAt'].apply(parse_hybrid_date)

/tmp/ipykernel_3718/812053565.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out7_cleaned['ActivityWeek'] = out7_cleaned['ActivityWeek'].apply(parse_hybrid_date)
/tmp/ipykernel_3718/812053565.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  out7_cleaned['LastActivityAt'] = out7_cleaned['LastActivityAt'].apply(parse_hybrid_date)


In [ ]:
out7_cleaned['Logins'] >= 0
out7_cl = out7_cleaned[out7_cleaned['Logins'] < 0]
out7_cl

,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
3451,ACT00003478,2024-05-21,STU000397,ENR000397,CRS0007,-3,4,3,3,2,2,1,2,39.8,105,1,0,2,0.0,2024-05-24 13:32:22
4147,ACT00004180,2024-07-03,STU000466,ENR000466,CRS0012,-10,5,3,2,2,1,1,3,65.4,205,2,2,2,10.8,2024-04-07 13:56:13
4400,ACT00004436,2024-02-29,STU000481,ENR000481,CRS0002,-6,1,0,0,1,0,0,0,0.0,96,0,2,3,2.9,2024-04-03 09:19:49
5986,ACT00006029,2024-03-21,STU000658,ENR000659,CRS0002,-4,3,2,1,1,0,0,1,58.3,175,0,2,0,3.8,2024-03-24 11:57:02
8039,ACT00008097,2024-10-09,STU000890,ENR000894,CRS0010,-8,3,2,1,2,2,1,1,92.7,103,0,1,1,6.1,2024-10-13 14:08:16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105714,ACT00106708,2025-12-10,STU010480,ENR010873,CRS0012,-5,5,1,1,1,1,1,2,60.2,159,1,2,4,9.1,2025-12-14 17:14:52
106069,ACT00107066,2025-12-24,STU010560,ENR010957,CRS0012,-3,1,0,0,3,1,1,3,36.8,0,1,0,1,0.0,2025-12-26 06:34:39
108804,ACT00109830,2025-12-17,STU011185,ENR011629,CRS0012,-2,3,2,2,3,1,1,1,35.9,16,0,2,0,0.0,2025-12-20 21:32:24
109196,ACT00110226,2025-12-29,STU011271,ENR011721,CRS0013,-10,3,3,2,2,2,2,2,90.6,95,2,1,2,5.8,2025-12-31 00:00:00


In [ ]:
out7_c2 = out7_cleaned[out7_cleaned['LessonsCompleted'] > out7_cleaned['LessonsViewed']]
out7_c2

,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
730,ACT00000735,2024-02-14,STU000083,ENR000083,CRS0012,5,1,-2,2,1,0,0,1,61.8,52,0,1,3,5.1,2024-02-17 07:10:47
766,ACT00000771,2024-02-27,STU000088,ENR000088,CRS0005,7,6,-3,2,1,0,0,2,55.6,162,0,2,1,6.7,2024-04-03 02:30:21
2749,ACT00002772,2024-06-05,STU000326,ENR000326,CRS0012,5,4,-4,4,1,1,1,4,41.4,181,1,3,3,9.3,2024-09-06 16:01:24
2773,ACT00002796,2024-05-08,STU000329,ENR000329,CRS0012,6,1,-3,2,2,1,1,2,168.5,124,1,0,2,4.7,2024-05-13 17:04:09
3740,ACT00003769,2024-07-03,STU000434,ENR000434,CRS0012,4,3,-7,0,1,0,0,2,36.1,63,0,0,2,2.1,2024-06-07 03:53:52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108599,ACT00109622,2025-11-18,STU011131,ENR011574,CRS0005,9,4,-3,2,2,2,1,4,54.2,81,0,2,1,3.7,2025-11-23 10:22:11
108988,ACT00110016,2025-12-10,STU011225,ENR011670,CRS0001,0,1,-7,0,1,0,0,0,0.0,14,1,0,0,0.0,2025-12-14 11:52:38
109802,ACT00110840,2025-12-07,STU011399,ENR011861,CRS0003,5,1,-8,0,3,1,1,0,0.0,62,1,0,0,4.4,2025-10-12 16:21:39
110450,ACT00111496,2025-12-04,STU011540,ENR012014,CRS0002,9,7,-2,1,3,1,1,1,73.3,93,2,2,4,8.3,2025-08-12 15:35:28


In [ ]:
out7_c3 = out7_cleaned[out7_cleaned['HomeworkAccepted'] > out7_cleaned['HomeworkSubmitted']]
out7_c3



,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
13,ACT00000014,2024-03-27,STU000002,ENR000002,CRS0001,2,1,0,0,3,1,4,0,0.0,25,0,1,1,0.2,2024-03-30 21:14:43
152,ACT00000155,2024-01-14,STU000019,ENR000019,CRS0008,4,1,0,0,2,1,2,1,33.6,0,0,0,2,2.5,2024-01-18 20:50:12
269,ACT00000273,2024-05-14,STU000032,ENR000032,CRS0007,4,3,0,0,3,2,3,0,0.0,40,1,1,1,3.1,2024-05-17 13:53:51
470,ACT00000475,2024-02-11,STU000055,ENR000055,CRS0008,5,2,3,3,1,1,2,3,56.4,51,1,2,1,6.3,2024-02-16 18:50:22
709,ACT00000714,2024-04-13,STU000082,ENR000082,CRS0017,2,1,0,0,2,1,2,0,0.0,15,1,0,0,0.0,2024-04-14 20:56:58
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110458,ACT00111504,2025-12-16,STU011541,ENR012015,CRS0005,10,7,3,3,2,2,5,0,0.0,126,2,0,3,5.4,2025-12-17 03:39:17
110703,ACT00111750,2025-12-21,STU011600,ENR012076,CRS0008,10,5,6,4,1,1,2,1,81.6,78,3,0,1,6.9,2025-12-26 16:29:37
111010,ACT00112062,2025-11-25,STU011665,ENR012149,CRS0005,7,4,3,2,2,1,4,2,66.2,93,2,2,1,7.1,2025-11-27 17:48:20
111026,ACT00112078,2025-12-14,STU011669,ENR012153,CRS0003,5,6,3,2,2,1,4,3,80.8,130,1,1,1,8.0,2025-12-15 23:32:50


In [ ]:
out7_c3 = out7_cleaned[out7_cleaned['AverageQuizScore'] > 100]
out7_c3

,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
145,ACT00000147,2024-03-11,STU000018,ENR000018,CRS0013,7,7,3,3,2,1,1,4,137.3,149,1,3,2,11.2,2024-03-13 09:53:00
334,ACT00000338,2024-04-10,STU000038,ENR000038,CRS0012,3,1,1,1,2,1,1,1,177.1,10,2,2,3,5.5,2024-04-15 22:37:06
426,ACT00000430,2024-03-27,STU000047,ENR000047,CRS0014,6,1,0,0,1,0,0,0,177.4,51,0,0,2,2.8,2024-03-31 02:55:06
461,ACT00000465,2024-04-01,STU000054,ENR000054,CRS0011,8,3,1,1,1,0,0,2,159.1,6,1,2,3,3.7,2024-04-04 07:03:15
745,ACT00000750,2024-04-03,STU000084,ENR000084,CRS0001,4,2,3,2,1,1,1,1,124.6,39,2,0,2,2.2,2024-07-04 13:20:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109423,ACT00110459,2025-12-15,STU011319,ENR011776,CRS0015,5,6,2,2,2,1,1,5,161.1,211,3,1,3,9.0,2025-12-21 00:07:28
110065,ACT00111108,2025-11-15,STU011458,ENR011924,CRS0006,4,3,3,3,3,2,4,3,143.9,126,1,1,3,3.9,2025-11-20 06:31:16
110066,ACT00111109,2025-11-22,STU011458,ENR011924,CRS0006,2,3,0,0,2,1,1,1,102.8,0,1,2,0,1.6,2025-11-23 01:31:45
110162,ACT00111206,2025-12-28,STU011479,ENR011945,CRS0003,0,0,0,0,1,0,0,0,173.2,24,1,1,1,0.0,2025-12-29 08:55:08


In [ ]:
out7_c4 = out7_cleaned[out7_cleaned['VideoMinutesWatched'] < 0]
out7_c4


,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
144,ACT00000146,2024-03-04,STU000018,ENR000018,CRS0013,8,7,6,4,2,2,2,3,95.3,-259,2,2,5,7.7,2024-08-03 18:59:48
4379,ACT00004415,2024-07-13,STU000479,ENR000479,CRS0017,3,0,1,1,3,1,1,1,54.3,-1,1,1,1,3.1,2024-07-16 14:37:09
5333,ACT00005371,2024-08-28,STU000588,ENR000588,CRS0010,2,1,0,0,2,0,0,0,0.0,-61,1,3,1,0.0,2024-02-09 05:09:15
9371,ACT00009439,2024-06-26,STU001014,ENR001018,CRS0012,1,3,0,0,1,0,0,0,0.0,-30,0,0,1,2.2,2024-06-30 12:02:32
9481,ACT00009551,2024-06-12,STU001023,ENR001027,CRS0012,1,3,2,1,3,2,1,0,0.0,-117,1,1,2,0.0,2024-06-16 18:29:31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100934,ACT00101882,2025-11-05,STU009593,ENR009931,CRS0012,9,7,3,2,3,2,2,5,81.8,-165,3,2,2,7.5,2025-10-11 08:31:15
101945,ACT00102903,2025-11-12,STU009741,ENR010090,CRS0004,0,2,1,1,3,1,1,0,0.0,-8,0,0,0,1.4,2025-11-18 05:04:57
104747,ACT00105735,2025-12-03,STU010285,ENR010661,CRS0010,9,7,4,4,2,1,1,4,96.4,-145,2,3,3,9.6,2025-06-12 11:47:59
106107,ACT00107105,2025-11-16,STU010569,ENR010967,CRS0008,10,7,3,2,1,1,1,3,61.1,-145,3,1,4,5.5,2025-11-19 12:01:12


In [ ]:
today = pd.Timestamp.now().normalize()

In [ ]:
future_dates_df = out7_cleaned[out7_cleaned['LastActivityAt'] > today]
future_dates_df

,ActivityID,ActivityWeek,StudentID,EnrollmentID,CourseID,Logins,ActiveDays,LessonsViewed,LessonsCompleted,HomeworkAssigned,HomeworkSubmitted,HomeworkAccepted,QuizAttempts,AverageQuizScore,VideoMinutesWatched,LiveLessonsAttended,QuestionsAsked,MentorMessages,PlatformHours,LastActivityAt
106,ACT00000107,2024-05-15,STU000009,ENR000009,CRS0014,1,1,1,1,3,1,1,0,0.0,68,2,0,0,0.4,2026-12-14 11:44:13
529,ACT00000534,2024-04-10,STU000062,ENR000062,CRS0012,10,7,2,1,2,1,1,3,78.8,204,2,1,0,5.9,2027-02-02 22:48:27
1266,ACT00001276,2024-03-19,STU000143,ENR000143,CRS0005,4,4,3,3,2,1,1,3,89.4,150,0,2,2,2.9,2026-10-11 16:17:18
2603,ACT00002624,2024-05-01,STU000312,ENR000312,CRS0014,6,5,2,2,2,1,1,1,44.2,134,0,0,1,1.3,2027-06-01 21:45:49
3039,ACT00003063,2024-05-04,STU000354,ENR000354,CRS0006,2,1,3,2,1,1,1,2,62.1,80,2,1,1,1.6,2026-12-31 14:34:09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108623,ACT00109647,2025-12-15,STU011136,ENR011579,CRS0015,10,3,4,4,1,1,1,2,38.0,90,1,2,4,4.6,2027-12-01 20:08:56
108955,ACT00109982,2025-11-12,STU011218,ENR011663,CRS0010,2,1,1,1,2,1,1,0,0.0,27,0,2,0,1.8,2027-01-15 11:01:56
110361,ACT00111407,2025-11-15,STU011522,ENR011994,CRS0006,11,7,3,2,1,1,1,3,87.1,196,2,1,2,9.3,2027-01-30 17:16:00
110426,ACT00111472,2025-12-14,STU005878,ENR012008,CRS0008,0,3,1,1,3,1,1,0,0.0,36,0,0,0,0.1,2026-11-25 10:34:54


In [ ]:
# =====================================================
# EXPORT
# =====================================================
out7_cleaned.to_csv("StudentActiity_cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out7_cleaned.shape)

DONE: (111074, 20)


In [ ]:
path8 = '/content/drive/MyDrive/ExchangeRates.csv'

In [ ]:
# =====================================================
# LOAD ExchangeRates
# =====================================================
df8 = pd.read_csv(path8, encoding="utf-8-sig", dtype=str)
df8
out8 = df8.copy()
out8

,Date,Currency,BaseCurrency,ExchangeRate
0,2024-01-01,UAH,UAH,1.0
1,2024-01-02,UAH,UAH,1.0
2,2024-01-03,UAH,UAH,1.0
3,2024-01-04,UAH,UAH,1.0
4,2024-01-05,UAH,UAH,1.0
...,...,...,...,...
3639,2025-12-27,CZK,UAH,1.8933
3640,2025-12-28,CZK,UAH,1.9025
3641,2025-12-29,CZK,UAH,1.8981
3642,2025-12-30,CZK,UAH,1.9023


Data Audit

In [ ]:
out8.shape

(3644, 4)

Checking for duplicates

In [ ]:
total_duplicates8 = out8.duplicated().sum()
total_duplicates8


np.int64(0)

In [ ]:
out8.columns = out8.columns.str.strip()

In [ ]:
print(out8.columns.tolist())

['Date', 'Currency', 'BaseCurrency', 'ExchangeRate']


Checking for missing values

In [ ]:
out8.isnull().sum()

,0
Date,0
Currency,0
BaseCurrency,0
ExchangeRate,0


In [ ]:
out8.dtypes

,0
Date,object
Currency,object
BaseCurrency,object
ExchangeRate,object


Part 2 Data Cleaning

Data Types and Dates



In [ ]:
out8['Date'] = pd.to_datetime(out8['Date'], errors='coerce')
out8['Currency'] = out8['Currency'].astype(str)
out8['BaseCurrency'] = out8['BaseCurrency'].astype(str)
out8['ExchangeRate'] = out8['ExchangeRate'].astype(float)

In [ ]:
out8['Currency'].unique()

array(['UAH', 'EUR', 'USD', 'PLN', 'CZK'], dtype=object)

In [ ]:
ex_invalid = out8[out8['ExchangeRate'] < 0]
ex_invalid

,Date,Currency,BaseCurrency,ExchangeRate


In [ ]:
# =====================================================
# EXPORT
# =====================================================
out8.to_csv("ExchangeRate_cleaned.csv", index=False, encoding="utf-8-sig")

print("DONE:", out8.shape)

DONE: (3644, 4)
